In [4]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/gemma-4-developer-agent/HARNESS_README.md
/kaggle/input/competitions/gemma-4-developer-agent/tasks.jsonl
/kaggle/input/competitions/gemma-4-developer-agent/sample_submission/eval_config.yaml
/kaggle/input/competitions/gemma-4-developer-agent/sample_submission/agent.yaml
/kaggle/input/competitions/gemma-4-developer-agent/sample_submission/configs/sampling.yaml
/kaggle/input/competitions/gemma-4-developer-agent/sample_submission/prompts/system.md
/kaggle/input/competitions/gemma-4-developer-agent/sample_submission/prompts/analyzer.md
/kaggle/input/competitions/gemma-4-developer-agent/sample_submission/sub_agents/code_analyzer.yaml
/kaggle/input/competitions/gemma-4-developer-agent/sample_submission/adapters/tool_lora/adapter_model.safetensors
/kaggle/input/competitions/gemma-4-developer-agent/sample_submission/adapters/tool_lora/adapter_config.json
/kaggle/input/competitions/gemma-4-developer-agent/sample_submission/adapters/main_lora/adapter_model.safetensors


Cell 2 — dataset discovery

In [5]:
from pathlib import Path

INPUT = Path("/kaggle/input")
WORKING = Path("/kaggle/working")

def discover_dataset_root():
    # Look recursively for tasks.jsonl
    matches = list(INPUT.rglob("tasks.jsonl"))
    for task_file in matches:
        root = task_file.parent
        if (root / "sample_submission").exists():
            return root
    raise FileNotFoundError("Could not find dataset root containing tasks.jsonl and sample_submission")

DATASET_ROOT = discover_dataset_root()

TASKS_PATH = DATASET_ROOT / "tasks.jsonl"
HARNESS_README = DATASET_ROOT / "HARNESS_README.md"
SAMPLE_SUBMISSION = DATASET_ROOT / "sample_submission"
SNAPSHOTS_DIR = DATASET_ROOT / "snapshots"
GRAPHS_DIR = DATASET_ROOT / "graphs"
EMBEDDINGS_DIR = DATASET_ROOT / "embeddings"
WHEELS_DIR = DATASET_ROOT / "wheels"
SANDBOX_DIR = DATASET_ROOT / "sandbox"
DOCKER_DIR = DATASET_ROOT / "docker"

print("DATASET_ROOT:", DATASET_ROOT)
print("TASKS_PATH exists:", TASKS_PATH.exists(), TASKS_PATH)
print("HARNESS_README exists:", HARNESS_README.exists(), HARNESS_README)
print("SAMPLE_SUBMISSION exists:", SAMPLE_SUBMISSION.exists(), SAMPLE_SUBMISSION)
print("SNAPSHOTS_DIR exists:", SNAPSHOTS_DIR.exists(), SNAPSHOTS_DIR)
print("GRAPHS_DIR exists:", GRAPHS_DIR.exists(), GRAPHS_DIR)
print("EMBEDDINGS_DIR exists:", EMBEDDINGS_DIR.exists(), EMBEDDINGS_DIR)
print("WHEELS_DIR exists:", WHEELS_DIR.exists(), WHEELS_DIR)
print("SANDBOX_DIR exists:", SANDBOX_DIR.exists(), SANDBOX_DIR)
print("DOCKER_DIR exists:", DOCKER_DIR.exists(), DOCKER_DIR)

DATASET_ROOT: /kaggle/input/competitions/gemma-4-developer-agent
TASKS_PATH exists: True /kaggle/input/competitions/gemma-4-developer-agent/tasks.jsonl
HARNESS_README exists: True /kaggle/input/competitions/gemma-4-developer-agent/HARNESS_README.md
SAMPLE_SUBMISSION exists: True /kaggle/input/competitions/gemma-4-developer-agent/sample_submission
SNAPSHOTS_DIR exists: True /kaggle/input/competitions/gemma-4-developer-agent/snapshots
GRAPHS_DIR exists: True /kaggle/input/competitions/gemma-4-developer-agent/graphs
EMBEDDINGS_DIR exists: True /kaggle/input/competitions/gemma-4-developer-agent/embeddings
WHEELS_DIR exists: True /kaggle/input/competitions/gemma-4-developer-agent/wheels
SANDBOX_DIR exists: True /kaggle/input/competitions/gemma-4-developer-agent/sandbox
DOCKER_DIR exists: True /kaggle/input/competitions/gemma-4-developer-agent/docker


Cell 3 — quick inventory

In [6]:
def preview_tree(root, limit=60):
    print(f"\nListing from: {root}")
    count = 0
    for p in sorted(root.rglob("*")):
        rel = p.relative_to(root)
        print(rel)
        count += 1
        if count >= limit:
            print("... truncated ...")
            break

preview_tree(DATASET_ROOT, limit=80)
preview_tree(SAMPLE_SUBMISSION, limit=80)


Listing from: /kaggle/input/competitions/gemma-4-developer-agent
HARNESS_README.md
docker
docker/Dockerfile.public
docker/Dockerfile.sandbox
docker/imp.py
docker/telnetlib.py
embeddings
embeddings/fastapi_016ab760faba1813ba235d5582a014a58bd2f417.npz
embeddings/fastapi_08924400c2c75533685148b029defcf6913f1def.npz
embeddings/fastapi_08b09e5236e315b6f10265ed229f130d4befb4ea.npz
embeddings/fastapi_0bdc3ca3732e3742e094bea2b320e9774659ac38.npz
embeddings/fastapi_0dd42b746e17d33a74ea267a62eace9ccdccf345.npz
embeddings/fastapi_16d75d90eb96976a57b94cc24e4018859cd54c4d.npz
embeddings/fastapi_1b3bea8b6be2220cc1544653ea785d2134fcb35b.npz
embeddings/fastapi_1e78a36b7310003f0ff634627d8a7bc53c6ccdee.npz
embeddings/fastapi_1fcec88ad2df4541b6a81c4786c4d039dc745e1e.npz
embeddings/fastapi_202b2d2f5f331db9102b5dbcef071a9e09bed10e.npz
embeddings/fastapi_22b02e26f9e8c7e32bd8266e2b0ebe8bb3a0db2b.npz
embeddings/fastapi_27c0f7e75fc0ce26ca75295d455eb36f0ca033d3.npz
embeddings/fastapi_2b212ddd7604891d42e2ffbafe

Cell 4 — load tasks.jsonl

In [7]:
df = pd.read_json(TASKS_PATH, lines=True, dtype=False)

print("Tasks:", len(df))
print("Columns:", list(df.columns))
print("Repos:", df["repo"].nunique() if "repo" in df.columns else "n/a")

df.head(3)

Tasks: 129
Columns: ['instance_id', 'repo', 'base_commit', 'patch', 'test_patch', 'problem_statement', 'hints_text', 'created_at']
Repos: 4


,instance_id,repo,base_commit,patch,test_patch,problem_statement,hints_text,created_at
0,fastapi_15661,fastapi/fastapi,ee22a4b8ca46dcce26c8c183afc4992a888d8be2,--- a/scripts/prepare_release.py\n+++ b/script...,--- a/tests/test_prepare_release.py\n+++ b/tes...,👷 Automate release preparation\n\n## Pull Requ...,,2026-05-31 16:00:39+00:00
1,fastapi_15588,fastapi/fastapi,cb83b83dcf78eab4ea17d504db5abcda705fbdc4,--- a/fastapi/sse.py\n+++ b/fastapi/sse.py\n@@...,--- a/tests/test_sse.py\n+++ b/tests/test_sse....,♻️ Validate Server Sent Event fields to avoid ...,,2026-05-23 17:23:05+00:00
2,fastapi_15589,fastapi/fastapi,22b02e26f9e8c7e32bd8266e2b0ebe8bb3a0db2b,--- a/fastapi/dependencies/utils.py\n+++ b/fas...,--- a/tests/test_query_cookie_header_model_ext...,♻️ Do not accept underscore headers when using...,,2026-05-23 18:35:06+00:00


Cell 5 — patch parser / localization labels

In [8]:
import re
from pathlib import Path

HUNK_RE = re.compile(r"^@@ -(\d+)(?:,(\d+))? \+(\d+)(?:,(\d+))? @@(.*)$")
SYMBOL_RE = re.compile(r"(?:async\s+)?(?:def|class)\s+([A-Za-z_]\w*)")

def _clean_path(p: str) -> str:
    p = p.split("\t")[0].strip()
    return p[2:] if p.startswith(("a/", "b/")) else p

def parse_patch(text: str):
    files, cur, old_path = [], None, None
    rem_old = rem_new = 0

    def new_file(path):
        f = {
            "path": path,
            "added": 0,
            "deleted": 0,
            "hunks": 0,
            "symbols": set(),
            "_hdr": False,
        }
        files.append(f)
        return f

    for line in (text or "").splitlines():
        if rem_old > 0 or rem_new > 0:
            tag = line[:1]
            if tag == "+":
                cur["added"] += 1
                rem_new -= 1
                m = SYMBOL_RE.match(line[1:].strip())
                if m:
                    cur["symbols"].add(m.group(1))
                continue
            if tag == "-":
                cur["deleted"] += 1
                rem_old -= 1
                m = SYMBOL_RE.match(line[1:].strip())
                if m:
                    cur["symbols"].add(m.group(1))
                continue
            if tag == " " or line == "":
                rem_old -= 1
                rem_new -= 1
                continue
            if tag == "\\":
                continue
            rem_old = rem_new = 0

        if line.startswith("diff --git "):
            m = re.match(r"diff --git a/(\S+) b/(\S+)", line)
            cur = new_file(m.group(2) if m else line[11:].strip())
            continue

        if line.startswith("--- "):
            old_path = _clean_path(line[4:])
            continue

        if line.startswith("+++ "):
            new_path = _clean_path(line[4:])
            if cur is None or cur["_hdr"]:
                cur = new_file(new_path)
            cur["_hdr"] = True
            cur["path"] = new_path if new_path != "/dev/null" else old_path
            continue

        m = HUNK_RE.match(line)
        if m and cur is not None:
            rem_old = int(m.group(2)) if m.group(2) is not None else 1
            rem_new = int(m.group(4)) if m.group(4) is not None else 1
            cur["hunks"] += 1
            sm = SYMBOL_RE.search(m.group(5) or "")
            if sm:
                cur["symbols"].add(sm.group(1))

    for f in files:
        f["symbols"] = sorted(f["symbols"])
        f.pop("_hdr", None)

    return files

def is_test_path(path: str) -> bool:
    p = path.lower()
    return (
        "/tests/" in p or
        "/test_" in p or
        p.endswith("_test.py") or
        p.endswith("conftest.py")
    )

def patch_features(row):
    files = parse_patch(row.get("patch", ""))
    src_files = [f for f in files if not is_test_path(f["path"])]
    return pd.Series({
        "gold_files": [f["path"] for f in src_files],
        "gold_symbols": sorted({s for f in src_files for s in f["symbols"]}),
        "n_files_changed": len(src_files),
        "n_hunks": sum(f["hunks"] for f in src_files),
        "lines_added": sum(f["added"] for f in src_files),
        "lines_deleted": sum(f["deleted"] for f in src_files),
    })

feat = df.apply(patch_features, axis=1)
df = pd.concat([df, feat], axis=1)
df["patch_churn"] = df["lines_added"] + df["lines_deleted"]
df["issue_words"] = df["problem_statement"].fillna("").str.split().str.len()

df[[
    "instance_id", "repo", "gold_files", "gold_symbols",
    "n_files_changed", "n_hunks", "patch_churn"
]].head(10)

,instance_id,repo,gold_files,gold_symbols,n_files_changed,n_hunks,patch_churn
0,fastapi_15661,fastapi/fastapi,[scripts/prepare_release.py],"[bump_version, current_version, get_current_ve...",1,1,216
1,fastapi_15588,fastapi/fastapi,[fastapi/sse.py],"[EventSourceResponse, ServerSentEvent, _check_...",1,2,23
2,fastapi_15589,fastapi/fastapi,[fastapi/dependencies/utils.py],[request_params_to_args],1,1,4
3,fastapi_15030,fastapi/fastapi,[docs_src/server_sent_events/tutorial001_py310...,"[EventSourceResponse, Item, Prompt, ServerSent...",9,17,560
4,fastapi_14962,fastapi/fastapi,[docs_src/custom_response/tutorial010_py310.py...,"[app, serialize, serialize_json, serialize_res...",3,6,55
5,fastapi_15280,fastapi/fastapi,"[docs_src/vibe/tutorial001_py310.py, fastapi/a...","[ai_vibes, trace_item, vibe]",2,3,72
6,fastapi_14964,fastapi/fastapi,[fastapi/responses.py],[],1,2,52
7,fastapi_15023,fastapi/fastapi,[docs_src/stream_data/tutorial002_py310.py],"[PNGStreamingResponse, stream_image_no_async_y...",1,1,26
8,fastapi_14986,fastapi/fastapi,"[fastapi/applications.py, fastapi/openapi/docs...","[_html_safe_json, get_swagger_ui_html, openapi]",2,4,34
9,fastapi_14978,fastapi/fastapi,[docs_src/strict_content_type/tutorial001_py31...,"[Item, __init__, add_api_route, app, create_it...",3,14,82


Cell 6 — EDA

In [9]:
print("Median patch churn:", int(df["patch_churn"].median()))
print("Single-file rate:", round((df["n_files_changed"] == 1).mean(), 3))
print("Multi-hunk rate:", round((df["n_hunks"] > 1).mean(), 3))

hot_files = (
    df[["repo", "gold_files"]]
    .explode("gold_files")
    .dropna()
    .groupby(["repo", "gold_files"])
    .size()
    .sort_values(ascending=False)
    .reset_index(name="tasks")
)

hot_files.head(20)

Median patch churn: 12
Single-file rate: 0.705
Multi-hunk rate: 0.55


,repo,gold_files,tasks
0,fastapi/fastapi,fastapi/dependencies/utils.py,27
1,fastapi/fastapi,fastapi/routing.py,16
2,fastapi/fastapi,fastapi/_compat/v2.py,13
3,fastapi/fastapi,fastapi/openapi/utils.py,12
4,fastapi/fastapi,fastapi/dependencies/models.py,7
5,Textualize/rich,rich/console.py,7
6,fastapi/fastapi,fastapi/applications.py,7
7,Textualize/rich,rich/cells.py,6
8,psf/requests,src/requests/utils.py,5
9,fastapi/fastapi,fastapi/params.py,5


Cell 7 — read harness

In [10]:
if HARNESS_README.exists():
    harness_text = HARNESS_README.read_text(encoding="utf-8", errors="ignore")
    print(harness_text[:6000])
else:
    print("HARNESS_README.md not found")

# `swegemma` Evaluation Harness & Competitor Guide

This document provides a complete technical reference for the **`swegemma`** evaluation harness and its companion libraries (**`adk-submission`** and **`adk-eval-core`**). It explains how competitor submissions are validated and compiled, how the harness orchestrates agent sessions and sandboxes, how the built-in tools behave, and how generated patches are extracted and graded.

---

## Table of Contents

1. [System Architecture & Library Stack](#1-system-architecture--library-stack)
2. [Submission Contract & Declarative Agent Schema (`adk-submission`)](#2-submission-contract--declarative-agent-schema-adk-submission)
3. [Model Registry, Single-Model Rule & LoRA Adapters on 4x L4 GPUs](#3-model-registry-single-model-rule--lora-adapters-on-4x-l4-gpus)
4. [Two-Phase Evaluation Lifecycle](#4-two-phase-evaluation-lifecycle)
5. [Harness-to-Agent Interaction Protocol](#5-harness-to-agent-interaction-protocol)
6. [Built-In Tools Reference (`s

Cell 8 — tool inventory

In [11]:
KNOWN_TOOLS = [
    "run_command",
    "submit_patch",
    "get_status",
    "read_file",
    "edit_file",
    "write_file",
    "get_code_neighbors",
    "search_similar_code",
    "get_code_subgraph",
]

tool_inventory = {tool: (tool in harness_text) for tool in KNOWN_TOOLS}
tool_inventory

{'run_command': True,
 'submit_patch': True,
 'get_status': True,
 'read_file': True,
 'edit_file': True,
 'write_file': True,
 'get_code_neighbors': True,
 'search_similar_code': True,
 'get_code_subgraph': True}

Cell 9 — baseline prompt text

In [12]:
MAX_SEARCH_STEPS = 6
MAX_FILE_READS = 6

BASELINE_SYSTEM_PROMPT = f"""
You are an offline software-engineering agent operating inside the Gemma 4 Developer Agent competition harness.

Your goal is to resolve the issue by producing the smallest correct source patch.

High-level policy:
1. Restate the bug in 1-3 sentences.
2. Localize before editing. Prefer semantic/code-graph evidence over broad wandering.
3. Read only the most relevant files and symbols.
4. Form a concrete defect hypothesis before making edits.
5. Prefer minimal, behavior-preserving fixes.
6. Avoid refactors, cleanup, or stylistic rewrites unless required for correctness.
7. Verify narrowly when possible using focused commands or reproductions.
8. If the location is still too uncertain after bounded search, avoid a low-confidence patch.

Search budget guidance:
- At most {MAX_SEARCH_STEPS} meaningful search steps before deciding.
- At most {MAX_FILE_READS} targeted file reads before switching from search to diagnosis.

Editing guidance:
- Preserve existing interfaces unless the issue clearly requires a change.
- Reuse existing helper patterns from nearby code.
- Do not modify tests unless the task truly requires it.

Verification guidance:
- If a narrow reproduction or focused test exists, use it first.
- Prefer high-signal checks over broad expensive runs.
- Remove temporary debugging artifacts before submitting.

Final output behavior:
- Only submit a patch once you believe it is the best bounded attempt.
- If no credible fix is found, prefer no patch over random edits.
""".strip()

print(BASELINE_SYSTEM_PROMPT[:2000])

You are an offline software-engineering agent operating inside the Gemma 4 Developer Agent competition harness.

Your goal is to resolve the issue by producing the smallest correct source patch.

High-level policy:
1. Restate the bug in 1-3 sentences.
2. Localize before editing. Prefer semantic/code-graph evidence over broad wandering.
3. Read only the most relevant files and symbols.
4. Form a concrete defect hypothesis before making edits.
5. Prefer minimal, behavior-preserving fixes.
6. Avoid refactors, cleanup, or stylistic rewrites unless required for correctness.
7. Verify narrowly when possible using focused commands or reproductions.
8. If the location is still too uncertain after bounded search, avoid a low-confidence patch.

Search budget guidance:
- At most 6 meaningful search steps before deciding.
- At most 6 targeted file reads before switching from search to diagnosis.

Editing guidance:
- Preserve existing interfaces unless the issue clearly requires a change.
- Reuse e

Cell 10 — copy sample_submission/ into working dir

In [13]:
# import shutil
# import zipfile

# SUBMISSION_DIR = WORKING / "submission"
# SUBMISSION_ZIP = WORKING / "submission.zip"

# if SUBMISSION_DIR.exists():
#     shutil.rmtree(SUBMISSION_DIR)

# shutil.copytree(SAMPLE_SUBMISSION, SUBMISSION_DIR)

# print("Copied sample_submission to:", SUBMISSION_DIR)

Cell 11 — inspect copied config files

In [14]:
# config_files = [p for p in SUBMISSION_DIR.rglob("*") if p.suffix.lower() in {".yaml", ".yml", ".md", ".txt"}]

# for p in config_files[:20]:
#     print("=" * 100)
#     print(p.relative_to(SUBMISSION_DIR))
#     try:
#         print(p.read_text(encoding="utf-8", errors="ignore")[:1500])
#     except Exception as e:
#         print("Unreadable:", e)

Cell 12 — replace prompt file

In [15]:
# prompt_candidates = [
#     p for p in SUBMISSION_DIR.rglob("*")
#     if p.is_file() and p.suffix.lower() in {".md", ".txt"} and "prompt" in str(p).lower()
# ]

# if not prompt_candidates:
#     fallback = SUBMISSION_DIR / "prompts" / "system.md"
#     fallback.parent.mkdir(parents=True, exist_ok=True)
#     fallback.write_text(BASELINE_SYSTEM_PROMPT, encoding="utf-8")
#     target_prompt = fallback
# else:
#     ranked = sorted(
#         prompt_candidates,
#         key=lambda p: (0 if p.name.lower() == "system.md" else 1, len(str(p)))
#     )
#     target_prompt = ranked[0]
#     target_prompt.write_text(BASELINE_SYSTEM_PROMPT, encoding="utf-8")

# print("Updated prompt file:", target_prompt)

Cell 13 — build submission.zip

In [16]:
# if SUBMISSION_ZIP.exists():
#     SUBMISSION_ZIP.unlink()

# with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
#     for path in sorted(SUBMISSION_DIR.rglob("*")):
#         if path.is_file():
#             zf.write(path, path.relative_to(SUBMISSION_DIR))

# print("Created:", SUBMISSION_ZIP)
# print("Size (MB):", round(SUBMISSION_ZIP.stat().st_size / (1024 * 1024), 3))

Cell 14 — Sanity check the final archive

In [17]:
# with zipfile.ZipFile(SUBMISSION_ZIP, "r") as zf:
#     names = zf.namelist()
#     print("Archive entries:", len(names))
#     print("First 50 entries:")
#     for name in names[:50]:
#         print("-", name)

# assert "agent.yaml" in names, "Expected agent.yaml at archive root"
# print("Sanity check passed: agent.yaml is present at zip root.")

Cell 15 — Graph loader + index builder

In [18]:
# Cell 15 — Robust graph loader / index builder (empty-file tolerant)
import json
import gzip
import re
from collections import defaultdict
from pathlib import Path

GRAPH_CACHE = {}
GRAPH_INDEX_CACHE = {}
BAD_GRAPH_PATHS = set()

STOPWORDS = {
    "the", "and", "for", "with", "from", "that", "this", "when", "where", "into",
    "should", "would", "could", "while", "does", "doesnt", "dont", "after", "before",
    "have", "has", "had", "been", "being", "then", "than", "them", "they", "their",
    "there", "about", "because", "issue", "error", "errors", "bug", "fails", "failure",
    "request", "response", "return", "returns", "support", "handle", "handling"
}

TOKEN_RE = re.compile(r"[A-Za-z_][A-Za-z0-9_]+")

def repo_short(repo_name: str) -> str:
    return repo_name.split("/")[-1]

def normalize_tokens(text: str):
    toks = [t.lower() for t in TOKEN_RE.findall(text or "")]
    toks = [t for t in toks if len(t) >= 3 and t not in STOPWORDS]
    return toks

def is_usable_graph_file(path: Path) -> bool:
    return path is not None and path.exists() and path.is_file() and path.stat().st_size > 0

def graph_candidates_for_row(row):
    repo_name = repo_short(row["repo"])
    base_commit = str(row["base_commit"])

    candidates = [
        GRAPHS_DIR / f"{repo_name}_{base_commit}.json",   # prefer commit graph
        GRAPHS_DIR / f"{row['instance_id']}.json",        # fallback task graph
    ]

    # also allow shortened commit fallback
    short = base_commit[:12]
    candidates.extend(sorted(GRAPHS_DIR.glob(f"{repo_name}_{short}*.json")))

    # remove duplicates while preserving order
    seen = set()
    uniq = []
    for p in candidates:
        if p not in seen:
            seen.add(p)
            uniq.append(p)
    return uniq

def graph_path_for_row(row):
    for p in graph_candidates_for_row(row):
        if is_usable_graph_file(p):
            return p
    return None

def inspect_file_header(path, n=120):
    path = Path(path)
    print("Path:", path)
    print("Exists:", path.exists())
    print("Size:", path.stat().st_size if path.exists() else "n/a", "bytes")
    if path.exists():
        raw = path.read_bytes()[:n]
        print("First bytes:", raw[:32])
        print("As text:", raw.decode("utf-8", errors="replace"))

def load_graph_json(path: Path):
    path = Path(path)

    if path in GRAPH_CACHE:
        return GRAPH_CACHE[path]

    raw = path.read_bytes()
    if len(raw) == 0:
        raise ValueError(f"Empty graph file: {path}")

    # gzip magic
    if raw[:2] == b"\x1f\x8b":
        raw = gzip.decompress(raw)

    # strip BOM
    if raw.startswith(b"\xef\xbb\xbf"):
        raw = raw[3:]

    text = raw.decode("utf-8", errors="replace").strip()
    if not text:
        raise ValueError(f"Decoded graph file is empty after stripping: {path}")

    if text.startswith("version https://git-lfs.github.com/spec/v1"):
        raise ValueError(f"Graph file looks like a Git LFS pointer: {path}")

    if text[0] in "{[":
        obj = json.loads(text)
        GRAPH_CACHE[path] = obj
        return obj

    # fallback: maybe first non-empty line is JSON
    lines = [line for line in text.splitlines() if line.strip()]
    if lines and lines[0][0] in "{[":
        obj = json.loads(lines[0])
        GRAPH_CACHE[path] = obj
        return obj

    raise ValueError(f"Unrecognized graph format: {path}\n{text[:200]}")

def build_graph_index(graph_obj: dict):
    nodes = graph_obj.get("nodes", [])
    edges = graph_obj.get("edges", graph_obj.get("links", []))

    node_rows = []
    node_by_id = {}
    adj = defaultdict(set)

    for node in nodes:
        nid = node.get("id") or node.get("name")
        if not nid:
            continue

        name = node.get("name", nid)
        text = node.get("text", "") or ""
        tail = str(nid).split(".")[-1]
        blob = f"{nid}\n{name}\n{text[:1200]}".lower()

        row = {
            "id": str(nid),
            "name": str(name),
            "tail": tail,
            "blob": blob,
            "preview": text[:220].replace("\n", " "),
        }
        node_rows.append(row)
        node_by_id[row["id"]] = row

    for edge in edges:
        src = edge.get("source")
        dst = edge.get("target")
        if src is None or dst is None:
            continue
        src, dst = str(src), str(dst)
        adj[src].add(dst)
        adj[dst].add(src)  # undirected expansion baseline

    return {
        "nodes": node_rows,
        "node_by_id": node_by_id,
        "adj": adj,
        "num_nodes": len(node_rows),
        "num_edges": len(edges),
    }

def get_graph_index(row):
    gpath = graph_path_for_row(row)
    if gpath is None:
        return None, None

    if gpath in BAD_GRAPH_PATHS:
        return None, None

    try:
        if gpath not in GRAPH_INDEX_CACHE:
            graph_obj = load_graph_json(gpath)
            GRAPH_INDEX_CACHE[gpath] = build_graph_index(graph_obj)
        return GRAPH_INDEX_CACHE[gpath], gpath
    except Exception as e:
        print(f"[graph-skip] {row['instance_id']} -> {gpath.name}: {e}")
        BAD_GRAPH_PATHS.add(gpath)
        return None, None

# quick sanity check
first_row = df.iloc[0]
graph_index, graph_path = get_graph_index(first_row)
print("Graph path:", graph_path)
if graph_index:
    print("Nodes:", graph_index["num_nodes"], "Edges:", graph_index["num_edges"])
    print("Sample node IDs:")
    for item in graph_index["nodes"][:5]:
        print("-", item["id"])
else:
    print("No usable graph found for first row")

Graph path: /kaggle/input/competitions/gemma-4-developer-agent/graphs/fastapi_ee22a4b8ca46dcce26c8c183afc4992a888d8be2.json
Nodes: 4263 Edges: 2430
Sample node IDs:
- path_operation_configuration.tutorial005_py310.Item
- path_operation_configuration.tutorial001_py310.Item
- path_operation_configuration.tutorial002_py310.Item
- path_operation_configuration.tutorial003_py310.Item
- path_operation_configuration.tutorial004_py310.Item


Cell 16 — Direct symbol localization from issue text

In [19]:
# Cell 16 — Direct symbol localization from issue text (safe)

def issue_query_tokens(row, max_tokens=24):
    text = (row.get("problem_statement", "") or "") + "\n" + (row.get("hints_text", "") or "")
    toks = normalize_tokens(text)
    seen = set()
    out = []
    for t in toks:
        if t not in seen:
            seen.add(t)
            out.append(t)
        if len(out) >= max_tokens:
            break
    return out

def lexical_node_score(node, q_tokens, query_lc):
    blob = node["blob"]
    name_lc = node["name"].lower()
    tail_lc = node["tail"].lower()

    overlap = sum(tok in blob for tok in q_tokens)
    if overlap == 0:
        return 0.0

    score = float(overlap)

    if tail_lc in query_lc:
        score += 3.0
    if any(tok == tail_lc for tok in q_tokens):
        score += 3.0
    if any(tok in name_lc for tok in q_tokens):
        score += 1.0

    return score

def direct_symbol_candidates(row, top_k=15):
    graph_index, gpath = get_graph_index(row)
    if graph_index is None:
        return []

    query_text = ((row.get("problem_statement", "") or "") + "\n" + (row.get("hints_text", "") or "")).lower()
    q_tokens = issue_query_tokens(row)

    scored = []
    for node in graph_index["nodes"]:
        s = lexical_node_score(node, q_tokens, query_text)
        if s > 0:
            scored.append({
                "id": node["id"],
                "tail": node["tail"],
                "name": node["name"],
                "score": round(s, 3),
                "preview": node["preview"],
            })

    scored = sorted(scored, key=lambda x: (-x["score"], x["id"]))
    return scored[:top_k]

def ranked_hits_gold_symbol(ranked_ids, gold_symbols):
    gold = set(gold_symbols or [])
    if not gold:
        return False
    for nid in ranked_ids:
        tail = nid.split(".")[-1]
        if tail in gold or nid in gold:
            return True
    return False

GRAPH_EVAL_N = 30
eligible = df[df["gold_symbols"].map(len) > 0].copy()
subset = eligible.sample(min(GRAPH_EVAL_N, len(eligible)), random_state=42)

direct_eval_rows = []
for _, row in subset.iterrows():
    ranked = direct_symbol_candidates(row, top_k=15)
    ranked_ids = [x["id"] for x in ranked]

    direct_eval_rows.append({
        "instance_id": row["instance_id"],
        "repo": row["repo"],
        "gold_symbols": row["gold_symbols"],
        "direct_hit@1": ranked_hits_gold_symbol(ranked_ids[:1], row["gold_symbols"]),
        "direct_hit@5": ranked_hits_gold_symbol(ranked_ids[:5], row["gold_symbols"]),
        "direct_hit@10": ranked_hits_gold_symbol(ranked_ids[:10], row["gold_symbols"]),
        "num_ranked": len(ranked_ids),
        "graph_available": len(ranked_ids) > 0,
    })

direct_eval = pd.DataFrame(direct_eval_rows)

print("Direct symbol localization baseline")
print("Graph available rate:", round(direct_eval["graph_available"].mean(), 3))
print(direct_eval[["direct_hit@1", "direct_hit@5", "direct_hit@10"]].mean().round(3))

direct_eval.head(10)

Direct symbol localization baseline
Graph available rate: 1.0
direct_hit@1     0.233
direct_hit@5     0.233
direct_hit@10    0.267
dtype: float64


,instance_id,repo,gold_symbols,direct_hit@1,direct_hit@5,direct_hit@10,num_ranked,graph_available
0,rich_4077,Textualize/rich,"[flush, isatty]",True,True,True,15,True
1,fastapi_14455,fastapi/fastapi,[get_openapi_security_definitions],False,False,False,15,True
2,fastapi_14430,fastapi/fastapi,[copy_field_info],False,False,False,15,True
3,fastapi_11194,fastapi/fastapi,[_extract_form_body],False,False,False,15,True
4,fastapi_14962,fastapi/fastapi,"[app, serialize, serialize_json, serialize_res...",True,True,True,15,True
5,fastapi_14482,fastapi/fastapi,"[ModelField, __post_init__, create_body_model]",False,False,False,15,True
6,fastapi_13713,fastapi/fastapi,"[Item, get_openapi, openapi]",False,False,False,15,True
7,rich_3480,Textualize/rich,"[append, append_text]",True,True,True,15,True
8,fastapi_12942,fastapi/fastapi,[field_annotation_is_complex],False,False,False,15,True
9,fastapi_14419,fastapi/fastapi,"[Dependant, _uses_scopes, get_dependant, get_p...",False,False,True,15,True


Cell 17 — Graph-neighbor expansion baseline + example inspection

In [20]:
# Cell 17 — Neighbor expansion over top symbols

def expanded_symbol_candidates(row, seed_top_k=8, final_top_k=25, neighbor_bonus=0.35):
    graph_index, gpath = get_graph_index(row)
    if graph_index is None:
        return []

    query_text = ((row.get("problem_statement", "") or "") + "\n" + (row.get("hints_text", "") or "")).lower()
    q_tokens = issue_query_tokens(row)

    direct = direct_symbol_candidates(row, top_k=seed_top_k)
    if not direct:
        return []

    combined = {}
    for item in direct:
        combined[item["id"]] = float(item["score"])

    # expand through undirected graph neighborhood
    for item in direct:
        nid = item["id"]
        seed_score = float(item["score"])
        for nb in graph_index["adj"].get(nid, []):
            nb_node = graph_index["node_by_id"].get(nb)
            if nb_node is None:
                continue

            local_score = lexical_node_score(nb_node, q_tokens, query_text)
            total_score = local_score + neighbor_bonus * seed_score
            combined[nb] = max(combined.get(nb, 0.0), total_score)

    ranked = []
    for nid, score in combined.items():
        node = graph_index["node_by_id"][nid]
        ranked.append({
            "id": nid,
            "tail": node["tail"],
            "name": node["name"],
            "score": round(score, 3),
            "preview": node["preview"],
        })

    ranked = sorted(ranked, key=lambda x: (-x["score"], x["id"]))
    return ranked[:final_top_k]

neighbor_eval_rows = []
for _, row in subset.iterrows():
    direct_ranked = direct_symbol_candidates(row, top_k=10)
    expanded_ranked = expanded_symbol_candidates(row, seed_top_k=8, final_top_k=25)

    direct_ids = [x["id"] for x in direct_ranked]
    expanded_ids = [x["id"] for x in expanded_ranked]

    neighbor_eval_rows.append({
        "instance_id": row["instance_id"],
        "repo": row["repo"],
        "direct_hit@10": ranked_hits_gold_symbol(direct_ids[:10], row["gold_symbols"]),
        "expanded_hit@10": ranked_hits_gold_symbol(expanded_ids[:10], row["gold_symbols"]),
        "expanded_hit@25": ranked_hits_gold_symbol(expanded_ids[:25], row["gold_symbols"]),
    })

neighbor_eval = pd.DataFrame(neighbor_eval_rows)

print("Direct vs graph-expanded localization")
print(pd.DataFrame({
    "direct_hit@10": [neighbor_eval["direct_hit@10"].mean()],
    "expanded_hit@10": [neighbor_eval["expanded_hit@10"].mean()],
    "expanded_hit@25": [neighbor_eval["expanded_hit@25"].mean()],
}).round(3))

# Inspect one example task
example_row = subset.iloc[0]
print("\nExample task:", example_row["instance_id"], "| repo:", example_row["repo"])
print("Gold symbols:", example_row["gold_symbols"])
print("\nIssue snippet:")
print((example_row["problem_statement"] or "")[:800])

example_candidates = expanded_symbol_candidates(example_row, seed_top_k=8, final_top_k=12)
pd.DataFrame(example_candidates)[["id", "tail", "score", "preview"]]

Direct vs graph-expanded localization
   direct_hit@10  expanded_hit@10  expanded_hit@25
0          0.267              0.4            0.433

Example task: rich_4077 | repo: Textualize/rich
Gold symbols: ['flush', 'isatty']

Issue snippet:
proxy isatty

Fixes https://github.com/Textualize/rich/issues/4041


,id,tail,score,preview
0,rich._null_file.NullFile.isatty,isatty,9.00,def isatty(self) -> bool: return False
1,rich.progress._Reader.isatty,isatty,9.00,def isatty(self) -> bool: return self....
2,tests.test_console.FakeFile.isatty,isatty,8.00,def isatty(self) -> bool: self.cal...
3,tests.test_console.FakeTTY.isatty,isatty,8.00,def isatty(self) -> bool: self.cal...
4,rich._null_file.NullFile,NullFile,6.15,class NullFile(IO[str]): None def clo...
5,rich.progress._Reader,_Reader,6.15,"class _Reader(RawIOBase, BinaryIO): """"""A r..."
6,tests.test_ansi.test_decode_newlines,test_decode_newlines,6.00,"def test_decode_newlines(): """"""Test newlin..."
7,tests.test_cells.test_nerd_font,test_nerd_font,6.00,"def test_nerd_font(): """"""Regression test f..."
8,tests.test_console.test_capture_and_record,test_capture_and_record,6.00,"def test_capture_and_record() -> None: """"""..."
9,tests.test_console.test_recording_no_stdout_an...,test_recording_no_stdout_and_no_stderr_files,6.00,def test_recording_no_stdout_and_no_stderr_fil...


Cell 18 — clean summary dashboard

In [21]:
print("=== PUBLIC TASK SUMMARY ===")
print("Tasks:", len(df))
print("Repos:", df["repo"].nunique())
print("Median patch churn:", int(df["patch_churn"].median()))
print("Single-file rate:", round((df["n_files_changed"] == 1).mean(), 3))
print("Multi-hunk rate:", round((df["n_hunks"] > 1).mean(), 3))

print("\n=== GRAPH LOCALIZATION SUMMARY ===")
print("Graph available rate:", round(direct_eval["graph_available"].mean(), 3))
print("Direct hit@1 :", round(direct_eval["direct_hit@1"].mean(), 3))
print("Direct hit@5 :", round(direct_eval["direct_hit@5"].mean(), 3))
print("Direct hit@10:", round(direct_eval["direct_hit@10"].mean(), 3))

print("\nExpanded hit@10:", round(neighbor_eval["expanded_hit@10"].mean(), 3))
print("Expanded hit@25:", round(neighbor_eval["expanded_hit@25"].mean(), 3))

print("\n=== HOT FILES ===")
display(hot_files.head(15))

=== PUBLIC TASK SUMMARY ===
Tasks: 129
Repos: 4
Median patch churn: 12
Single-file rate: 0.705
Multi-hunk rate: 0.55

=== GRAPH LOCALIZATION SUMMARY ===
Graph available rate: 1.0
Direct hit@1 : 0.233
Direct hit@5 : 0.233
Direct hit@10: 0.267

Expanded hit@10: 0.4
Expanded hit@25: 0.433

=== HOT FILES ===


,repo,gold_files,tasks
0,fastapi/fastapi,fastapi/dependencies/utils.py,27
1,fastapi/fastapi,fastapi/routing.py,16
2,fastapi/fastapi,fastapi/_compat/v2.py,13
3,fastapi/fastapi,fastapi/openapi/utils.py,12
4,fastapi/fastapi,fastapi/dependencies/models.py,7
5,Textualize/rich,rich/console.py,7
6,fastapi/fastapi,fastapi/applications.py,7
7,Textualize/rich,rich/cells.py,6
8,psf/requests,src/requests/utils.py,5
9,fastapi/fastapi,fastapi/params.py,5


Cell 19 — inspect failed cases

In [22]:
from IPython.display import display

failed_ids = neighbor_eval.loc[~neighbor_eval["expanded_hit@25"], "instance_id"].tolist()

print("Failed cases at expanded_hit@25:", len(failed_ids))
print("Showing first 5 failures...\n")

for iid in failed_ids[:5]:
    row = subset[subset["instance_id"] == iid].iloc[0]
    cands = expanded_symbol_candidates(row, seed_top_k=8, final_top_k=12)

    print("=" * 120)
    print("INSTANCE:", row["instance_id"])
    print("REPO:", row["repo"])
    print("GOLD SYMBOLS:", row["gold_symbols"])
    print("\nISSUE SNIPPET:")
    print((row["problem_statement"] or "")[:700])

    if len(cands) == 0:
        print("\nNo candidates returned.\n")
        continue

    print("\nTOP CANDIDATES:")
    display(pd.DataFrame(cands)[["id", "tail", "score", "preview"]])

Failed cases at expanded_hit@25: 17
Showing first 5 failures...

INSTANCE: fastapi_14455
REPO: fastapi/fastapi
GOLD SYMBOLS: ['get_openapi_security_definitions']

ISSUE SNIPPET:
🐛 Fix OpenAPI security scheme OAuth2 scopes declaration, deduplicate security schemes with different scopes

🐛 Fix OpenAPI security scheme OAuth2 scopes declaration, deduplicate security schemes with different scopes

This should fix https://github.com/fastapi/fastapi/issues/14454

TOP CANDIDATES:


,id,tail,score,preview
0,fastapi.security.oauth2.OAuth2,OAuth2,17.15,"class OAuth2(SecurityBase): """"""This is the..."
1,fastapi.param_functions.Security,Security,16.15,def Security( # noqa: N802 dependency: An...
2,fastapi.applications.FastAPI.openapi,openapi,14.50,"def openapi(self) -> Dict[str, Any]: ""..."
3,fastapi.security.oauth2.OAuth2PasswordBearer,OAuth2PasswordBearer,13.90,"class OAuth2PasswordBearer(OAuth2): """"""OAu..."
4,fastapi.applications.FastAPI,FastAPI,13.85,"class FastAPI(Starlette): """"""`FastAPI` app..."
5,fastapi.params.Security,Security,13.55,@dataclass(frozen=True) class Security(Depends...
6,fastapi.security.oauth2.OAuth2.make_not_authen...,make_not_authenticated_error,11.90,def make_not_authenticated_error(self) -> HTTP...
7,fastapi.security.oauth2.OAuth2AuthorizationCod...,OAuth2AuthorizationCodeBearer,11.90,class OAuth2AuthorizationCodeBearer(OAuth2): ...
8,fastapi.openapi.models.OAuth2,OAuth2,11.00,class OAuth2(SecurityBase): None
9,fastapi.security.oauth2.OAuth2.__init__,__init__,10.90,"def __init__( self, *, ..."


INSTANCE: fastapi_14430
REPO: fastapi/fastapi
GOLD SYMBOLS: ['copy_field_info']

ISSUE SNIPPET:
🐛 Fix optional sequence handling with new union syntax from Python 3.10

https://github.com/fastapi/fastapi/pull/14297 is only taking one union type into account. I can only recommend using https://github.com/pydantic/typing-inspection for these kind of checks (see https://typing-inspection.pydantic.dev/latest/usage/ for motivation).

TOP CANDIDATES:


,id,tail,score,preview
0,fastapi.applications.FastAPI,FastAPI,16.50,"class FastAPI(Starlette): """"""`FastAPI` app..."
1,fastapi.applications.FastAPI.include_router,include_router,14.55,"def include_router( self, rout..."
2,fastapi.applications.FastAPI.websocket,websocket,14.55,"def websocket( self, path: Ann..."
3,fastapi.applications.FastAPI.middleware,middleware,12.55,"def middleware( self, middlewa..."
4,fastapi.applications.FastAPI.add_api_route,add_api_route,11.55,"def add_api_route( self, path:..."
5,fastapi.applications.FastAPI.api_route,api_route,11.55,"def api_route( self, path: str..."
6,fastapi.applications.FastAPI.exception_handler,exception_handler,11.55,"def exception_handler( self, e..."
7,tests.test_compat.test_propagates_pydantic2_mo...,test_propagates_pydantic2_model_config,11.55,def test_propagates_pydantic2_model_config(): ...
8,fastapi.routing.APIRouter,APIRouter,11.50,"class APIRouter(routing.Router): """"""`APIRo..."
9,fastapi.applications.FastAPI.__init__,__init__,10.55,"def __init__( self: AppType, *..."


INSTANCE: fastapi_11194
REPO: fastapi/fastapi
GOLD SYMBOLS: ['_extract_form_body']

ISSUE SNIPPET:
🐛 Fix validation error when `File` is declared after `Form` parameter

This PR updates the fix (6185a92) from @jsg921019 so that tests pass with the latest FastAPI code (0.110.0).
It also adds a test to cover the case with multiple files, and another to ensure the order of parameters in an endpoint definition does not affect the way FastAPI handles the request.

**Problem**
When declaring an endpoint which accepts a `File` but also a `Form` parameter, the `File` parameter must be declared first or the endpoint would respond with error 422.

**Context**
- See discussion #9116
- And probably also #4625


TOP CANDIDATES:


,id,tail,score,preview
0,fastapi.params.File,File,16.20,class File(Form): None def __init__(s...
1,fastapi.params.Form,Form,16.20,class Form(Body): None def __init__(s...
2,fastapi.param_functions.File,File,15.20,def File( # noqa: N802 default: Annotated...
3,fastapi.param_functions.Form,Form,15.20,def Form( # noqa: N802 default: Annotated...
4,tests.test_tutorial.test_background_tasks.test...,test,12.00,"def test(): log = Path(""log.txt"") if l..."
5,tests.test_tutorial.test_background_tasks.test...,test,12.00,"def test(client: TestClient): log = Path(""..."
6,tests.test_tutorial.test_cookie_params.test_tu...,test,10.00,"def test(path, cookies, expected_status, expec..."
7,tests.test_tutorial.test_extending_openapi.tes...,test,10.00,"def test(): response = client.get(""/items/..."
8,fastapi.security.oauth2.OAuth2PasswordRequestF...,__init__,9.85,"def __init__( self, grant_type..."
9,fastapi.params.File.__init__,__init__,9.20,"def __init__( self, default: A..."


INSTANCE: fastapi_14482
REPO: fastapi/fastapi
GOLD SYMBOLS: ['ModelField', '__post_init__', 'create_body_model']

ISSUE SNIPPET:
🐛 Fix handling arbitrary types when using `arbitrary_types_allowed=True`

🐛 Fix handling arbitrary types when using `arbitrary_types_allowed=True`

Related to https://github.com/fastapi/fastapi/discussions/14184, https://github.com/fastapi/fastapi/issues/14483



TOP CANDIDATES:


,id,tail,score,preview
0,fastapi.applications.FastAPI,FastAPI,12.45,"class FastAPI(Starlette): """"""`FastAPI` app..."
1,tests.test_compat.test_propagates_pydantic2_mo...,test_propagates_pydantic2_model_config,10.50,def test_propagates_pydantic2_model_config(): ...
2,fastapi.applications.FastAPI.include_router,include_router,9.50,"def include_router( self, rout..."
3,tests.test_inherited_custom_class.test_pydanticv1,test_pydanticv1,9.50,def test_pydanticv1(): app = FastAPI() ...
4,tests.test_inherited_custom_class.test_pydanticv2,test_pydanticv2,9.50,def test_pydanticv2(): from pydantic impor...
5,fastapi.applications.FastAPI.websocket,websocket,8.50,"def websocket( self, path: Ann..."
6,scripts.people.Discussions,Discussions,8.00,class Discussions(BaseModel): None
7,fastapi.applications.FastAPI.__init__,__init__,7.50,"def __init__( self: AppType, *..."
8,fastapi.applications.FastAPI.delete,delete,7.50,"def delete( self, path: Annota..."
9,fastapi.applications.FastAPI.exception_handler,exception_handler,7.50,"def exception_handler( self, e..."


INSTANCE: fastapi_13713
REPO: fastapi/fastapi
GOLD SYMBOLS: ['Item', 'get_openapi', 'openapi']

ISSUE SNIPPET:
✨ Add OpenAPI `external_docs` parameter to `FastAPI`

This PR adds an external_docs parameter to the __init__ method of the FastAPI class. Currently, the external_docs parameter is not being passed when generating the OpenAPI documentation, which I believe is an oversight. By including this parameter in the FastAPI class constructor, the external_docs field will be correctly included in the root of the generated OpenAPI JSON or YAML file.

Reasoning:

The external_docs field provides valuable external documentation links and should be part of the OpenAPI metadata. Including it at the root level of the OpenAPI spec enhances the usability and completeness of the generated documentation, es

TOP CANDIDATES:


,id,tail,score,preview
0,fastapi.security.api_key.APIKeyQuery,APIKeyQuery,14.90,"class APIKeyQuery(APIKeyBase): """"""API key ..."
1,fastapi.security.api_key.APIKeyQuery.__init__,__init__,14.00,"def __init__( self, *, ..."
2,fastapi.routing.APIRouter,APIRouter,13.55,"class APIRouter(routing.Router): """"""`APIRo..."
3,fastapi.security.api_key.APIKeyCookie,APIKeyCookie,13.55,"class APIKeyCookie(APIKeyBase): """"""API key..."
4,fastapi.security.api_key.APIKeyHeader,APIKeyHeader,13.55,"class APIKeyHeader(APIKeyBase): """"""API key..."
5,fastapi.routing.APIRouter.__init__,__init__,13.00,"def __init__( self, *, ..."
6,fastapi.security.api_key.APIKeyCookie.__init__,__init__,13.00,"def __init__( self, *, ..."
7,fastapi.security.api_key.APIKeyHeader.__init__,__init__,13.00,"def __init__( self, *, ..."
8,fastapi.security.http.HTTPBasic.__init__,__init__,13.00,"def __init__( self, *, ..."
9,fastapi.security.http.HTTPBearer.__init__,__init__,13.00,"def __init__( self, *, ..."


Cell 20 — source-first reranking

In [23]:
def repo_prefix_for_row(row):
    mapping = {
        "fastapi/fastapi": "fastapi.",
        "Textualize/rich": "rich.",
        "psf/requests": "requests.",
        "encode/httpx": "httpx.",
    }
    return mapping.get(row["repo"], "")

def is_test_node_id(nid: str) -> bool:
    nid = nid.lower()
    return (
        nid.startswith("tests.")
        or ".tests." in nid
        or ".test_" in nid
        or "conftest" in nid
    )

def source_first_rerank(row, candidates):
    prefix = repo_prefix_for_row(row)
    rescored = []

    for item in candidates:
        score2 = float(item["score"])
        nid = item["id"]

        if prefix and nid.startswith(prefix):
            score2 += 0.75

        if is_test_node_id(nid):
            score2 -= 2.5

        rescored.append({
            **item,
            "score2": round(score2, 3),
        })

    rescored = sorted(rescored, key=lambda x: (-x["score2"], x["id"]))
    return rescored

def expanded_symbol_candidates_source_first(row, seed_top_k=8, final_top_k=25):
    base = expanded_symbol_candidates(row, seed_top_k=seed_top_k, final_top_k=final_top_k)
    if not base:
        return []
    return source_first_rerank(row, base)[:final_top_k]

source_eval_rows = []
for _, row in subset.iterrows():
    ranked = expanded_symbol_candidates_source_first(row, seed_top_k=8, final_top_k=25)
    ranked_ids = [x["id"] for x in ranked]

    source_eval_rows.append({
        "instance_id": row["instance_id"],
        "repo": row["repo"],
        "source_first_hit@10": ranked_hits_gold_symbol(ranked_ids[:10], row["gold_symbols"]),
        "source_first_hit@25": ranked_hits_gold_symbol(ranked_ids[:25], row["gold_symbols"]),
    })

source_eval = pd.DataFrame(source_eval_rows)

print("=== SOURCE-FIRST RERANK RESULTS ===")
print(pd.DataFrame({
    "expanded_hit@10": [neighbor_eval["expanded_hit@10"].mean()],
    "expanded_hit@25": [neighbor_eval["expanded_hit@25"].mean()],
    "source_first_hit@10": [source_eval["source_first_hit@10"].mean()],
    "source_first_hit@25": [source_eval["source_first_hit@25"].mean()],
}).round(3))

=== SOURCE-FIRST RERANK RESULTS ===
   expanded_hit@10  expanded_hit@25  source_first_hit@10  source_first_hit@25
0              0.4            0.433                  0.4                0.433


Cell 21 — TF-IDF retriever over graph nodes

In [24]:
# Cell 21 — TF-IDF retriever over graph node blobs
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

TFIDF_CACHE = {}

def get_tfidf_index(row):
    graph_index, gpath = get_graph_index(row)
    if graph_index is None:
        return None, None, None

    if gpath in TFIDF_CACHE:
        return TFIDF_CACHE[gpath]

    texts = [node["blob"] for node in graph_index["nodes"]]
    if len(texts) == 0:
        TFIDF_CACHE[gpath] = (None, None, graph_index)
        return TFIDF_CACHE[gpath]

    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        min_df=1,
        sublinear_tf=True,
    )
    X = vectorizer.fit_transform(texts)
    TFIDF_CACHE[gpath] = (vectorizer, X, graph_index)
    return TFIDF_CACHE[gpath]

def tfidf_symbol_candidates(row, top_k=15, return_score_map=False):
    vectorizer, X, graph_index = get_tfidf_index(row)
    if vectorizer is None or X is None or graph_index is None:
        return ({}, []) if return_score_map else []

    query_text = (row.get("problem_statement", "") or "") + "\n" + (row.get("hints_text", "") or "")
    q = vectorizer.transform([query_text])

    if q.nnz == 0:
        return ({}, []) if return_score_map else []

    scores = (X @ q.T).toarray().ravel()
    order = np.argsort(-scores)

    ranked = []
    score_map = {}
    for idx in order:
        s = float(scores[idx])
        if s <= 0:
            break
        node = graph_index["nodes"][idx]
        score_map[node["id"]] = s
        ranked.append({
            "id": node["id"],
            "tail": node["tail"],
            "name": node["name"],
            "score": round(s, 4),
            "preview": node["preview"],
        })
        if len(ranked) >= top_k:
            break

    return (score_map, ranked) if return_score_map else ranked

# quick smoke test
smoke_row = subset.iloc[0]
tfidf_test = tfidf_symbol_candidates(smoke_row, top_k=10)
pd.DataFrame(tfidf_test)[["id", "tail", "score"]]

,id,tail,score
0,tests.test_text.test_wrap_invalid_style,test_wrap_invalid_style,0.3255
1,tests.test_cells.test_nerd_font,test_nerd_font,0.2894
2,tests.test_text.test_wrap_cjk_mixed,test_wrap_cjk_mixed,0.2886
3,tests.test_text.test_wrap_long_words_2,test_wrap_long_words_2,0.2807
4,tests.test_text.test_append_loop_regression,test_append_loop_regression,0.2687
5,tests.test_traceback.test_guess_lexer_yaml_j2,test_guess_lexer_yaml_j2,0.2679
6,tests.test_pretty.test_measure_pretty,test_measure_pretty,0.2386
7,tests.test_style.test_clear_meta_and_links_cle...,test_clear_meta_and_links_clears_hash,0.2380
8,tests.test_traceback.test_recursive_exception,test_recursive_exception,0.2331
9,tests.test_syntax.test_padding_plus_wrap,test_padding_plus_wrap,0.2289


Cell 22 — TF-IDF + graph hybrid expansion

In [25]:
# Cell 22 — TF-IDF seed retrieval + graph expansion

def hybrid_tfidf_graph_candidates(row, seed_top_k=8, final_top_k=25, neighbor_bonus=0.35):
    graph_index, gpath = get_graph_index(row)
    if graph_index is None:
        return []

    score_map, seeds = tfidf_symbol_candidates(row, top_k=seed_top_k, return_score_map=True)
    if not seeds:
        return []

    combined = {}

    # keep seed scores
    for item in seeds:
        combined[item["id"]] = float(item["score"])

    # expand neighbors
    for item in seeds:
        nid = item["id"]
        seed_score = float(item["score"])
        for nb in graph_index["adj"].get(nid, []):
            nb_base = float(score_map.get(nb, 0.0))
            total = nb_base + neighbor_bonus * seed_score
            combined[nb] = max(combined.get(nb, 0.0), total)

    ranked = []
    for nid, score in combined.items():
        node = graph_index["node_by_id"].get(nid)
        if node is None:
            continue
        ranked.append({
            "id": nid,
            "tail": node["tail"],
            "name": node["name"],
            "score": round(score, 4),
            "preview": node["preview"],
        })

    ranked = sorted(ranked, key=lambda x: (-x["score"], x["id"]))
    return ranked[:final_top_k]

hybrid_eval_rows = []
for _, row in subset.iterrows():
    tfidf_ranked = tfidf_symbol_candidates(row, top_k=10)
    hybrid_ranked = hybrid_tfidf_graph_candidates(row, seed_top_k=8, final_top_k=25)

    tfidf_ids = [x["id"] for x in tfidf_ranked]
    hybrid_ids = [x["id"] for x in hybrid_ranked]

    hybrid_eval_rows.append({
        "instance_id": row["instance_id"],
        "repo": row["repo"],
        "tfidf_hit@10": ranked_hits_gold_symbol(tfidf_ids[:10], row["gold_symbols"]),
        "hybrid_hit@10": ranked_hits_gold_symbol(hybrid_ids[:10], row["gold_symbols"]),
        "hybrid_hit@25": ranked_hits_gold_symbol(hybrid_ids[:25], row["gold_symbols"]),
        "tfidf_candidates": len(tfidf_ids),
        "hybrid_candidates": len(hybrid_ids),
    })

hybrid_eval = pd.DataFrame(hybrid_eval_rows)

print("=== TF-IDF + GRAPH HYBRID RESULTS ===")
print(pd.DataFrame({
    "graph_direct_hit@10": [direct_eval["direct_hit@10"].mean()],
    "graph_expanded_hit@10": [neighbor_eval["expanded_hit@10"].mean()],
    "graph_expanded_hit@25": [neighbor_eval["expanded_hit@25"].mean()],
    "tfidf_hit@10": [hybrid_eval["tfidf_hit@10"].mean()],
    "hybrid_hit@10": [hybrid_eval["hybrid_hit@10"].mean()],
    "hybrid_hit@25": [hybrid_eval["hybrid_hit@25"].mean()],
}).round(3))

=== TF-IDF + GRAPH HYBRID RESULTS ===
   graph_direct_hit@10  graph_expanded_hit@10  graph_expanded_hit@25  \
0                0.267                    0.4                  0.433   

   tfidf_hit@10  hybrid_hit@10  hybrid_hit@25  
0         0.267          0.267            0.3  


Cell 23 — inspect hybrid failures / wins

In [26]:
# Cell 23 — Inspect where hybrid helps or still fails

compare = neighbor_eval.merge(
    hybrid_eval[["instance_id", "tfidf_hit@10", "hybrid_hit@10", "hybrid_hit@25"]],
    on="instance_id",
    how="left"
)

print("Cases where hybrid improved over graph-expanded@10:")
improved = compare[(compare["expanded_hit@10"] == False) & (compare["hybrid_hit@10"] == True)]
print(len(improved))
display(improved.head(10))

print("\nCases still failing at hybrid_hit@25:")
still_fail = compare[compare["hybrid_hit@25"] == False]
print(len(still_fail))
display(still_fail.head(10))

# show one improved example and one failed example
for label, frame in [("IMPROVED", improved), ("STILL_FAIL", still_fail)]:
    if len(frame) == 0:
        continue

    iid = frame.iloc[0]["instance_id"]
    row = subset[subset["instance_id"] == iid].iloc[0]
    cands = hybrid_tfidf_graph_candidates(row, seed_top_k=8, final_top_k=12)

    print("\n" + "=" * 120)
    print(label, "| INSTANCE:", row["instance_id"], "| REPO:", row["repo"])
    print("GOLD SYMBOLS:", row["gold_symbols"])
    print("\nISSUE SNIPPET:")
    print((row["problem_statement"] or "")[:700])

    if len(cands):
        display(pd.DataFrame(cands)[["id", "tail", "score", "preview"]])

Cases where hybrid improved over graph-expanded@10:
3


,instance_id,repo,direct_hit@10,expanded_hit@10,expanded_hit@25,tfidf_hit@10,hybrid_hit@10,hybrid_hit@25
9,fastapi_14419,fastapi/fastapi,True,False,False,True,True,True
14,rich_3454,Textualize/rich,False,False,False,True,True,True
25,fastapi_14458,fastapi/fastapi,False,False,False,True,True,True



Cases still failing at hybrid_hit@25:
21


,instance_id,repo,direct_hit@10,expanded_hit@10,expanded_hit@25,tfidf_hit@10,hybrid_hit@10,hybrid_hit@25
0,rich_4077,Textualize/rich,True,True,True,False,False,False
1,fastapi_14455,fastapi/fastapi,False,False,False,False,False,False
2,fastapi_14430,fastapi/fastapi,False,False,False,False,False,False
3,fastapi_11194,fastapi/fastapi,False,False,False,False,False,False
4,fastapi_14962,fastapi/fastapi,True,True,True,False,False,False
5,fastapi_14482,fastapi/fastapi,False,False,False,False,False,False
7,rich_3480,Textualize/rich,True,True,True,False,False,False
8,fastapi_12942,fastapi/fastapi,False,False,False,False,False,False
10,fastapi_14356,fastapi/fastapi,False,False,False,False,False,False
11,rich_3064,Textualize/rich,False,True,True,False,False,False



IMPROVED | INSTANCE: fastapi_14419 | REPO: fastapi/fastapi
GOLD SYMBOLS: ['Dependant', '_uses_scopes', 'get_dependant', 'get_parameterless_sub_dependant', 'oauth_scopes', 'solve_dependencies']

ISSUE SNIPPET:
🐛 Cache dependencies that don't use scopes and don't have sub-dependencies with scopes

🐛 Cache dependencies that don't use scopes and don't have sub-dependencies with scopes

Related to https://github.com/fastapi/fastapi/pull/9790, and https://github.com/fastapi/fastapi/discussions/6024

Uses the ideas in @YuriiMotov's diagrams here for the tests: https://github.com/fastapi/fastapi/discussions/6024#discussioncomment-8541913

---

Dependencies are cached based on the function. And also on the **scopes**, but (now) **only** when those scopes are used by this dependency or by a sub-dependency: this dependency or a sub-dependency declare scopes or access the `SecurityScopes`.


,id,tail,score,preview
0,fastapi.security.oauth2.SecurityScopes,SecurityScopes,0.2413,"class SecurityScopes: """"""This is a special..."
1,fastapi.security.oauth2.SecurityScopes.__init__,__init__,0.2102,"def __init__( self, scopes: An..."
2,tests.test_get_model_definitions_formfeed_esca...,test_model_description_escaped_with_formfeed,0.1583,def test_model_description_escaped_with_formfe...
3,tests.test_tutorial.test_custom_request_and_ro...,test_exception_handler_body_access,0.1205,def test_exception_handler_body_access(): ...
4,fastapi.param_functions.Security,Security,0.1135,def Security( # noqa: N802 dependency: An...
5,scripts.people.Discussions,Discussions,0.1132,class Discussions(BaseModel): None
6,fastapi.security.oauth2.OAuth2PasswordBearer,OAuth2PasswordBearer,0.1107,"class OAuth2PasswordBearer(OAuth2): """"""OAu..."
7,fastapi.dependencies.utils.get_parameterless_s...,get_parameterless_sub_dependant,0.1045,"def get_parameterless_sub_dependant(*, depends..."
8,fastapi._compat.v1.get_model_definitions,get_model_definitions,0.0554,"def get_model_definitions( *, flat_mod..."
9,fastapi.openapi.utils.get_fields_from_routes,get_fields_from_routes,0.0554,def get_fields_from_routes( routes: Sequen...



STILL_FAIL | INSTANCE: rich_4077 | REPO: Textualize/rich
GOLD SYMBOLS: ['flush', 'isatty']

ISSUE SNIPPET:
proxy isatty

Fixes https://github.com/Textualize/rich/issues/4041


,id,tail,score,preview
0,tests.test_text.test_wrap_invalid_style,test_wrap_invalid_style,0.3255,def test_wrap_invalid_style(): # https://g...
1,tests.test_cells.test_nerd_font,test_nerd_font,0.2894,"def test_nerd_font(): """"""Regression test f..."
2,tests.test_text.test_wrap_cjk_mixed,test_wrap_cjk_mixed,0.2886,"def test_wrap_cjk_mixed(): """"""Regression t..."
3,tests.test_text.test_wrap_long_words_2,test_wrap_long_words_2,0.2807,def test_wrap_long_words_2(): # https://gi...
4,tests.test_text.test_append_loop_regression,test_append_loop_regression,0.2687,def test_append_loop_regression() -> None: ...
5,tests.test_traceback.test_guess_lexer_yaml_j2,test_guess_lexer_yaml_j2,0.2679,def test_guess_lexer_yaml_j2(): # https://...
6,tests.test_pretty.test_measure_pretty,test_measure_pretty,0.2386,"def test_measure_pretty() -> None: """"""Test..."
7,tests.test_style.test_clear_meta_and_links_cle...,test_clear_meta_and_links_clears_hash,0.2380,def test_clear_meta_and_links_clears_hash(): ...
8,rich.console.Console,Console,0.1139,"class Console: """"""A high level console int..."
9,rich.cells.cell_len,cell_len,0.1013,"def cell_len(text: str, unicode_version: str =..."


Cell 24 — Clean query + source-only TF-IDF index

In [27]:
# Cell 24 — Clean query + source-first TF-IDF
import re
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

TFIDF2_CACHE = {}

def clean_query_text(row):
    text = (row.get("problem_statement", "") or "") + "\n" + (row.get("hints_text", "") or "")
    text = re.sub(r"https?://\S+", " ", text)      # remove URLs
    text = text.replace("`", " ")
    text = text.replace("**", " ")
    text = re.sub(r"#+", " ", text)                # markdown headings
    text = re.sub(r"\s+", " ", text).strip()
    return text

def repo_prefix_for_row(row):
    mapping = {
        "fastapi/fastapi": "fastapi.",
        "Textualize/rich": "rich.",
        "psf/requests": "requests.",
        "encode/httpx": "httpx.",
    }
    return mapping.get(row["repo"], "")

def is_test_node_id(nid: str) -> bool:
    nid = nid.lower()
    return (
        nid.startswith("tests.")
        or ".tests." in nid
        or ".test_" in nid
        or "conftest" in nid
    )

def is_preferred_source_node(row, nid: str) -> bool:
    prefix = repo_prefix_for_row(row)
    if is_test_node_id(nid):
        return False
    if prefix and nid.startswith(prefix):
        return True
    return not is_test_node_id(nid)

def get_tfidf_index_source_first(row, source_only=True):
    graph_index, gpath = get_graph_index(row)
    if graph_index is None:
        return None, None, None, None

    cache_key = (gpath, "source_only" if source_only else "all_nodes")
    if cache_key in TFIDF2_CACHE:
        return TFIDF2_CACHE[cache_key]

    selected_nodes = []
    for node in graph_index["nodes"]:
        nid = node["id"]
        if source_only:
            if not is_preferred_source_node(row, nid):
                continue
        selected_nodes.append(node)

    texts = [node["blob"] for node in selected_nodes]
    if len(texts) == 0:
        TFIDF2_CACHE[cache_key] = (None, None, None, graph_index)
        return TFIDF2_CACHE[cache_key]

    vectorizer = TfidfVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        min_df=1,
        sublinear_tf=True,
    )
    X = vectorizer.fit_transform(texts)

    TFIDF2_CACHE[cache_key] = (vectorizer, X, selected_nodes, graph_index)
    return TFIDF2_CACHE[cache_key]

def tfidf_symbol_candidates_source_first(row, top_k=15, source_only=True, return_score_map=False):
    vectorizer, X, selected_nodes, graph_index = get_tfidf_index_source_first(row, source_only=source_only)
    if vectorizer is None or X is None or selected_nodes is None:
        return ({}, []) if return_score_map else []

    query_text = clean_query_text(row)
    q = vectorizer.transform([query_text])

    if q.nnz == 0:
        return ({}, []) if return_score_map else []

    scores = (X @ q.T).toarray().ravel()
    order = np.argsort(-scores)

    ranked = []
    score_map = {}
    for idx in order:
        s = float(scores[idx])
        if s <= 0:
            break
        node = selected_nodes[idx]
        nid = node["id"]
        score_map[nid] = s
        ranked.append({
            "id": nid,
            "tail": node["tail"],
            "name": node["name"],
            "score": round(s, 4),
            "preview": node["preview"],
        })
        if len(ranked) >= top_k:
            break

    return (score_map, ranked) if return_score_map else ranked

# smoke test on prior bad case
row = subset[subset["instance_id"] == "rich_4077"].iloc[0]
clean_tfidf_test = tfidf_symbol_candidates_source_first(row, top_k=10, source_only=True)
pd.DataFrame(clean_tfidf_test)[["id", "tail", "score"]]

,id,tail,score
0,rich.progress._Reader.isatty,isatty,0.3288
1,rich._null_file.NullFile.isatty,isatty,0.2966
2,rich.jupyter.print,print,0.1482
3,rich._null_file.NullFile,NullFile,0.0543
4,rich.progress._Reader,_Reader,0.0522


Cell 25 — Clean source-first TF-IDF + graph expansion

In [28]:
# Cell 25 — Clean source-first TF-IDF + graph expansion

def hybrid_clean_source_graph_candidates(row, seed_top_k=8, final_top_k=25, neighbor_bonus=0.35):
    graph_index, gpath = get_graph_index(row)
    if graph_index is None:
        return []

    score_map, seeds = tfidf_symbol_candidates_source_first(
        row,
        top_k=seed_top_k,
        source_only=True,
        return_score_map=True,
    )
    if not seeds:
        return []

    combined = {}

    for item in seeds:
        combined[item["id"]] = float(item["score"])

    for item in seeds:
        nid = item["id"]
        seed_score = float(item["score"])
        for nb in graph_index["adj"].get(nid, []):
            nb_node = graph_index["node_by_id"].get(nb)
            if nb_node is None:
                continue
            if is_test_node_id(nb):
                continue

            nb_base = float(score_map.get(nb, 0.0))
            total = nb_base + neighbor_bonus * seed_score

            # tiny repo-native bias
            prefix = repo_prefix_for_row(row)
            if prefix and nb.startswith(prefix):
                total += 0.05

            combined[nb] = max(combined.get(nb, 0.0), total)

    ranked = []
    for nid, score in combined.items():
        node = graph_index["node_by_id"].get(nid)
        if node is None:
            continue
        ranked.append({
            "id": nid,
            "tail": node["tail"],
            "name": node["name"],
            "score": round(score, 4),
            "preview": node["preview"],
        })

    ranked = sorted(ranked, key=lambda x: (-x["score"], x["id"]))
    return ranked[:final_top_k]

clean_hybrid_eval_rows = []
for _, row in subset.iterrows():
    clean_tfidf_ranked = tfidf_symbol_candidates_source_first(row, top_k=10, source_only=True)
    clean_hybrid_ranked = hybrid_clean_source_graph_candidates(row, seed_top_k=8, final_top_k=25)

    clean_tfidf_ids = [x["id"] for x in clean_tfidf_ranked]
    clean_hybrid_ids = [x["id"] for x in clean_hybrid_ranked]

    clean_hybrid_eval_rows.append({
        "instance_id": row["instance_id"],
        "repo": row["repo"],
        "clean_tfidf_hit@10": ranked_hits_gold_symbol(clean_tfidf_ids[:10], row["gold_symbols"]),
        "clean_hybrid_hit@10": ranked_hits_gold_symbol(clean_hybrid_ids[:10], row["gold_symbols"]),
        "clean_hybrid_hit@25": ranked_hits_gold_symbol(clean_hybrid_ids[:25], row["gold_symbols"]),
    })

clean_hybrid_eval = pd.DataFrame(clean_hybrid_eval_rows)

print("=== CLEAN SOURCE-FIRST TF-IDF + GRAPH RESULTS ===")
print(pd.DataFrame({
    "graph_expanded_hit@10": [neighbor_eval["expanded_hit@10"].mean()],
    "graph_expanded_hit@25": [neighbor_eval["expanded_hit@25"].mean()],
    "clean_tfidf_hit@10": [clean_hybrid_eval["clean_tfidf_hit@10"].mean()],
    "clean_hybrid_hit@10": [clean_hybrid_eval["clean_hybrid_hit@10"].mean()],
    "clean_hybrid_hit@25": [clean_hybrid_eval["clean_hybrid_hit@25"].mean()],
}).round(3))

=== CLEAN SOURCE-FIRST TF-IDF + GRAPH RESULTS ===
   graph_expanded_hit@10  graph_expanded_hit@25  clean_tfidf_hit@10  \
0                    0.4                  0.433               0.433   

   clean_hybrid_hit@10  clean_hybrid_hit@25  
0                0.467                0.467  


Cell 26 — Inspect if the cleaned retriever fixed the earlier bad example

In [29]:
# Cell 26 — Inspect cleaned hybrid on previous failure examples

inspect_ids = [
    "rich_4077",
    "fastapi_14455",
    "fastapi_14430",
    "fastapi_11194",
    "fastapi_14482",
]

for iid in inspect_ids:
    rows = subset[subset["instance_id"] == iid]
    if len(rows) == 0:
        continue
    row = rows.iloc[0]

    cands = hybrid_clean_source_graph_candidates(row, seed_top_k=8, final_top_k=12)

    print("\n" + "=" * 120)
    print("INSTANCE:", row["instance_id"])
    print("REPO:", row["repo"])
    print("GOLD SYMBOLS:", row["gold_symbols"])
    print("\nCLEAN QUERY:")
    print(clean_query_text(row)[:500])

    if len(cands) == 0:
        print("No candidates returned.")
        continue

    display(pd.DataFrame(cands)[["id", "tail", "score", "preview"]])


INSTANCE: rich_4077
REPO: Textualize/rich
GOLD SYMBOLS: ['flush', 'isatty']

CLEAN QUERY:
proxy isatty Fixes


,id,tail,score,preview
0,rich.progress._Reader.isatty,isatty,0.3971,def isatty(self) -> bool: return self....
1,rich._null_file.NullFile.isatty,isatty,0.3657,def isatty(self) -> bool: return False
2,rich.progress._Reader,_Reader,0.2172,"class _Reader(RawIOBase, BinaryIO): """"""A r..."
3,rich._null_file.NullFile,NullFile,0.2081,class NullFile(IO[str]): None def clo...
4,rich.jupyter.print,print,0.1482,"def print(*args: Any, **kwargs: Any) -> None: ..."
5,rich.get_console,get_console,0.1019,"def get_console() -> ""Console"": """"""Get a g..."
6,rich._null_file.NullFile.__enter__,__enter__,0.0690,def __enter__(self) -> IO[str]: return...
7,rich._null_file.NullFile.__exit__,__exit__,0.0690,"def __exit__( self, __t: Optio..."
8,rich._null_file.NullFile.__iter__,__iter__,0.0690,def __iter__(self) -> Iterator[str]: r...
9,rich._null_file.NullFile.__next__,__next__,0.0690,"def __next__(self) -> str: return """""



INSTANCE: fastapi_14455
REPO: fastapi/fastapi
GOLD SYMBOLS: ['get_openapi_security_definitions']

CLEAN QUERY:
🐛 Fix OpenAPI security scheme OAuth2 scopes declaration, deduplicate security schemes with different scopes 🐛 Fix OpenAPI security scheme OAuth2 scopes declaration, deduplicate security schemes with different scopes This should fix


,id,tail,score,preview
0,fastapi.security.oauth2.SecurityScopes,SecurityScopes,0.4268,"class SecurityScopes: """"""This is a special..."
1,fastapi.security.oauth2.OAuth2PasswordBearer._...,__init__,0.3812,"def __init__( self, tokenUrl: ..."
2,fastapi.security.oauth2.OAuth2AuthorizationCod...,__init__,0.3646,"def __init__( self, authorizat..."
3,fastapi.security.oauth2.OAuth2PasswordBearer,OAuth2PasswordBearer,0.3593,"class OAuth2PasswordBearer(OAuth2): """"""OAu..."
4,fastapi.security.oauth2.OAuth2AuthorizationCod...,OAuth2AuthorizationCodeBearer,0.3476,class OAuth2AuthorizationCodeBearer(OAuth2): ...
5,fastapi.security.oauth2.SecurityScopes.__init__,__init__,0.3295,"def __init__( self, scopes: An..."
6,fastapi.security.oauth2.OAuth2.__init__,__init__,0.2634,"def __init__( self, *, ..."
7,fastapi.openapi.models.OAuth2,OAuth2,0.2547,class OAuth2(SecurityBase): None
8,fastapi.openapi.models.OAuthFlows,OAuthFlows,0.1389,class OAuthFlows(BaseModelWithConfig): None
9,fastapi.security.oauth2.OAuth2,OAuth2,0.1271,"class OAuth2(SecurityBase): """"""This is the..."



INSTANCE: fastapi_14430
REPO: fastapi/fastapi
GOLD SYMBOLS: ['copy_field_info']

CLEAN QUERY:
🐛 Fix optional sequence handling with new union syntax from Python 3.10 is only taking one union type into account. I can only recommend using for these kind of checks (see for motivation).


,id,tail,score,preview
0,fastapi.routing.APIRoute.__init__,__init__,0.1939,"def __init__( self, path: str,..."
1,fastapi.routing.APIRoute,APIRoute,0.1879,class APIRoute(routing.Route): None d...
2,fastapi.dependencies.utils.is_union_of_base_mo...,is_union_of_base_models,0.1455,def is_union_of_base_models(field_type: Any) -...
3,fastapi.routing.APIRouter.add_api_route,add_api_route,0.1061,"def add_api_route( self, path:..."
4,fastapi.applications.FastAPI.add_api_route,add_api_route,0.1049,"def add_api_route( self, path:..."
5,fastapi._compat.main._is_model_class,_is_model_class,0.1009,def _is_model_class(value: Any) -> bool: i...
6,fastapi.dependencies.utils._should_embed_body_...,_should_embed_body_fields,0.1009,def _should_embed_body_fields(fields: List[Mod...
7,fastapi._compat.shared.lenient_issubclass,lenient_issubclass,0.0882,"def lenient_issubclass( cls: Any, class_or..."
8,fastapi.datastructures.Default,Default,0.0882,def Default(value: DefaultType) -> DefaultType...
9,fastapi.dependencies.utils.get_body_field,get_body_field,0.0882,"def get_body_field( *, flat_dependant: Dep..."



INSTANCE: fastapi_11194
REPO: fastapi/fastapi
GOLD SYMBOLS: ['_extract_form_body']

CLEAN QUERY:
🐛 Fix validation error when File is declared after Form parameter This PR updates the fix (6185a92) from @jsg921019 so that tests pass with the latest FastAPI code (0.110.0). It also adds a test to cover the case with multiple files, and another to ensure the order of parameters in an endpoint definition does not affect the way FastAPI handles the request. Problem When declaring an endpoint which accepts a File but also a Form parameter, the File parameter must be declared first or the endpoint 


,id,tail,score,preview
0,fastapi.param_functions.File,File,0.2165,def File( # noqa: N802 default: Annotated...
1,fastapi.params.File,File,0.1799,class File(Form): None def __init__(s...
2,fastapi.param_functions.Body,Body,0.1362,def Body( # noqa: N802 default: Annotated...
3,fastapi.param_functions.Form,Form,0.1204,def Form( # noqa: N802 default: Annotated...
4,fastapi.openapi.models.Parameter,Parameter,0.1187,class Parameter(ParameterBase): None
5,fastapi.params.Body,Body,0.0977,class Body(FieldInfo): None def __ini...
6,fastapi.datastructures.UploadFile,UploadFile,0.0975,"class UploadFile(StarletteUploadFile): """"""..."
7,fastapi.params.Form,Form,0.0921,class Form(Body): None def __init__(s...
8,fastapi.security.oauth2.OAuth2PasswordRequestF...,__init__,0.0921,"def __init__( self, *, ..."
9,fastapi.security.oauth2.OAuth2PasswordRequestF...,__init__,0.0921,"def __init__( self, grant_type..."



INSTANCE: fastapi_14482
REPO: fastapi/fastapi
GOLD SYMBOLS: ['ModelField', '__post_init__', 'create_body_model']

CLEAN QUERY:
🐛 Fix handling arbitrary types when using arbitrary_types_allowed=True 🐛 Fix handling arbitrary types when using arbitrary_types_allowed=True Related to


,id,tail,score,preview
0,fastapi.applications.FastAPI,FastAPI,0.0765,"class FastAPI(Starlette): """"""`FastAPI` app..."
1,fastapi.exceptions.HTTPException.__init__,__init__,0.0758,"def __init__( self, status_cod..."
2,fastapi.security.http.HTTPBase.make_not_authen...,make_not_authenticated_error,0.0758,def make_not_authenticated_error(self) -> HTTP...
3,fastapi.security.oauth2.OAuth2.make_not_authen...,make_not_authenticated_error,0.0758,def make_not_authenticated_error(self) -> HTTP...
4,fastapi.applications.FastAPI.exception_handler,exception_handler,0.0756,"def exception_handler( self, e..."
5,fastapi.exceptions.HTTPException,HTTPException,0.0737,class HTTPException(StarletteHTTPException): ...
6,fastapi._compat.main._is_model_class,_is_model_class,0.0723,def _is_model_class(value: Any) -> bool: i...
7,fastapi.dependencies.utils._should_embed_body_...,_should_embed_body_fields,0.0723,def _should_embed_body_fields(fields: List[Mod...
8,fastapi.params.Header,Header,0.0656,class Header(Param): None def __init_...
9,fastapi.temp_pydantic_v1_params.Header,Header,0.0656,class Header(Param): None def __init_...


Cell 27 — Snapshot file catalog + symbol → file mapping

In [30]:
# Cell 27 — Robust snapshot python-file catalog + symbol -> file mapping
import tarfile
from pathlib import Path
from collections import defaultdict

SNAPSHOT_FILE_CACHE = {}

def get_snapshot_path(row):
    return SNAPSHOTS_DIR / f"{row['instance_id']}.tgz"

def all_python_suffixes(path_str: str):
    """
    For an archive member path like:
      some/root/fastapi/applications.py
    generate:
      some/root/fastapi/applications.py
      root/fastapi/applications.py
      fastapi/applications.py
      applications.py
    """
    p = Path(path_str)
    parts = p.parts
    out = set()

    for i in range(len(parts)):
        suffix = Path(*parts[i:])
        if suffix.suffix == ".py":
            out.add(str(suffix))

    return out

def get_snapshot_py_files(row):
    """
    Returns a set of normalized .py file paths extracted from the snapshot tar.
    Includes all suffix variants so we can match module paths robustly.
    """
    tpath = get_snapshot_path(row)
    if tpath in SNAPSHOT_FILE_CACHE:
        return SNAPSHOT_FILE_CACHE[tpath]

    py_files = set()

    if not tpath.exists():
        SNAPSHOT_FILE_CACHE[tpath] = py_files
        return py_files

    with tarfile.open(tpath, "r:gz") as tf:
        for m in tf.getmembers():
            if not m.isfile():
                continue
            name = m.name
            if not name.endswith(".py"):
                continue
            py_files.update(all_python_suffixes(name))

    SNAPSHOT_FILE_CACHE[tpath] = py_files
    return py_files

def symbol_id_to_module_candidates(symbol_id: str):
    """
    Convert a graph symbol id like:
      fastapi.dependencies.utils.get_dependant
    into likely module file paths:
      fastapi/dependencies/utils.py
      fastapi/dependencies/utils/__init__.py
      fastapi/dependencies.py
      ...
    """
    parts = str(symbol_id).split(".")
    candidates = []

    # progressively shorten from full dotted name toward package/module root
    for k in range(len(parts) - 1, 0, -1):
        mod = "/".join(parts[:k])
        candidates.append(f"{mod}.py")
        candidates.append(f"{mod}/__init__.py")

    # dedupe preserve order
    seen = set()
    ordered = []
    for c in candidates:
        if c not in seen:
            seen.add(c)
            ordered.append(c)
    return ordered

def symbol_id_to_candidate_files(row, symbol_id):
    py_files = get_snapshot_py_files(row)
    raw_candidates = symbol_id_to_module_candidates(symbol_id)

    hits = []

    # exact match first
    for c in raw_candidates:
        if c in py_files:
            hits.append(c)

    if hits:
        # dedupe preserve order
        out = []
        seen = set()
        for h in hits:
            if h not in seen:
                seen.add(h)
                out.append(h)
        return out

    # fallback: suffix match against normalized snapshot files
    for c in raw_candidates:
        for f in py_files:
            if f == c or f.endswith("/" + c):
                hits.append(f)

    out = []
    seen = set()
    for h in hits:
        if h not in seen:
            seen.add(h)
            out.append(h)
    return out

def symbols_to_ranked_files(row, symbol_candidates, max_files=10):
    ranked_files = []
    seen_files = set()

    for item in symbol_candidates:
        sid = item["id"]
        score = item["score"]
        cands = symbol_id_to_candidate_files(row, sid)

        for f in cands:
            if f not in seen_files:
                seen_files.add(f)
                ranked_files.append({
                    "file": f,
                    "from_symbol": sid,
                    "score": score,
                })

        if len(ranked_files) >= max_files:
            break

    return ranked_files[:max_files]

# quick debug on a few known cases
for iid in ["rich_4077", "fastapi_14455", "fastapi_14962"]:
    rows = subset[subset["instance_id"] == iid]
    if len(rows) == 0:
        continue
    row = rows.iloc[0]
    print("\n" + "=" * 100)
    print("INSTANCE:", iid)
    print("GOLD FILES:", row["gold_files"])

    cands = hybrid_clean_source_graph_candidates(row, seed_top_k=8, final_top_k=8)
    print("Top symbol IDs:")
    for x in cands[:5]:
        print("-", x["id"])

    mapped = symbols_to_ranked_files(row, cands, max_files=8)
    print("Mapped files:")
    for m in mapped:
        print("-", m["file"], "| from:", m["from_symbol"])


INSTANCE: rich_4077
GOLD FILES: ['rich/file_proxy.py']
Top symbol IDs:
- rich.progress._Reader.isatty
- rich._null_file.NullFile.isatty
- rich.progress._Reader
- rich._null_file.NullFile
- rich.jupyter.print
Mapped files:
- rich/progress.py | from: rich.progress._Reader.isatty
- rich/__init__.py | from: rich.progress._Reader.isatty
- rich/_null_file.py | from: rich._null_file.NullFile.isatty
- rich/jupyter.py | from: rich.jupyter.print

INSTANCE: fastapi_14455
GOLD FILES: ['fastapi/openapi/utils.py']
Top symbol IDs:
- fastapi.security.oauth2.SecurityScopes
- fastapi.security.oauth2.OAuth2PasswordBearer.__init__
- fastapi.security.oauth2.OAuth2AuthorizationCodeBearer.__init__
- fastapi.security.oauth2.OAuth2PasswordBearer
- fastapi.security.oauth2.OAuth2AuthorizationCodeBearer
Mapped files:
- fastapi/security/oauth2.py | from: fastapi.security.oauth2.SecurityScopes
- fastapi/security/__init__.py | from: fastapi.security.oauth2.SecurityScopes
- fastapi/__init__.py | from: fastapi.securi

Cell 28 — File recall@1/@3/@5 from improved hybrid retriever

In [31]:
# Cell 28 — File-level recall evaluation

def ranked_hits_gold_file(ranked_files, gold_files):
    gold = set(gold_files or [])
    return any(f in gold for f in ranked_files)

file_eval_rows = []

for _, row in subset.iterrows():
    symbol_ranked = hybrid_clean_source_graph_candidates(row, seed_top_k=8, final_top_k=25)
    file_ranked = symbols_to_ranked_files(row, symbol_ranked, max_files=10)
    file_ids = [x["file"] for x in file_ranked]

    file_eval_rows.append({
        "instance_id": row["instance_id"],
        "repo": row["repo"],
        "gold_files": row["gold_files"],
        "pred_files": file_ids,
        "file_hit@1": ranked_hits_gold_file(file_ids[:1], row["gold_files"]),
        "file_hit@3": ranked_hits_gold_file(file_ids[:3], row["gold_files"]),
        "file_hit@5": ranked_hits_gold_file(file_ids[:5], row["gold_files"]),
        "file_hit@10": ranked_hits_gold_file(file_ids[:10], row["gold_files"]),
        "num_pred_files": len(file_ids),
    })

file_eval = pd.DataFrame(file_eval_rows)

print("=== FILE-LEVEL RECALL FROM CLEAN HYBRID SYMBOL RETRIEVER ===")
print(pd.DataFrame({
    "file_hit@1": [file_eval["file_hit@1"].mean()],
    "file_hit@3": [file_eval["file_hit@3"].mean()],
    "file_hit@5": [file_eval["file_hit@5"].mean()],
    "file_hit@10": [file_eval["file_hit@10"].mean()],
    "avg_pred_files": [file_eval["num_pred_files"].mean()],
}).round(3))

file_eval.head(10)

=== FILE-LEVEL RECALL FROM CLEAN HYBRID SYMBOL RETRIEVER ===
   file_hit@1  file_hit@3  file_hit@5  file_hit@10  avg_pred_files
0         0.2         0.3         0.4        0.433             7.8


,instance_id,repo,gold_files,pred_files,file_hit@1,file_hit@3,file_hit@5,file_hit@10,num_pred_files
0,rich_4077,Textualize/rich,[rich/file_proxy.py],"[rich/progress.py, rich/__init__.py, rich/_nul...",False,False,False,False,4
1,fastapi_14455,fastapi/fastapi,[fastapi/openapi/utils.py],"[fastapi/security/oauth2.py, fastapi/security/...",False,False,False,False,5
2,fastapi_14430,fastapi/fastapi,[fastapi/_compat/v2.py],"[fastapi/routing.py, fastapi/__init__.py, fast...",False,False,False,False,10
3,fastapi_11194,fastapi/fastapi,[fastapi/dependencies/utils.py],"[fastapi/param_functions.py, fastapi/__init__....",False,False,False,True,10
4,fastapi_14962,fastapi/fastapi,[docs_src/custom_response/tutorial010_py310.py...,"[fastapi/applications.py, fastapi/__init__.py,...",False,True,True,True,4
5,fastapi_14482,fastapi/fastapi,[fastapi/_compat/v2.py],"[fastapi/applications.py, fastapi/__init__.py,...",False,False,False,False,10
6,fastapi_13713,fastapi/fastapi,"[fastapi/applications.py, fastapi/openapi/util...","[fastapi/openapi/models.py, fastapi/openapi/__...",False,False,False,False,10
7,rich_3480,Textualize/rich,[rich/text.py],"[rich/protocol.py, rich/__init__.py, rich/cons...",False,False,False,False,10
8,fastapi_12942,fastapi/fastapi,[fastapi/_compat.py],"[fastapi/security/api_key.py, fastapi/security...",False,False,False,False,9
9,fastapi_14419,fastapi/fastapi,"[fastapi/dependencies/models.py, fastapi/depen...","[fastapi/security/oauth2.py, fastapi/security/...",False,False,True,True,10


Cell 29 — Inspect file-level misses

In [32]:
# Cell 29 — Inspect file-level failures

file_fail = file_eval[~file_eval["file_hit@5"]].copy()

print("File-level misses at @5:", len(file_fail))
display(file_fail[["instance_id", "repo", "gold_files", "pred_files"]].head(10))

for iid in file_fail["instance_id"].head(5):
    row = subset[subset["instance_id"] == iid].iloc[0]
    symbol_ranked = hybrid_clean_source_graph_candidates(row, seed_top_k=8, final_top_k=12)
    file_ranked = symbols_to_ranked_files(row, symbol_ranked, max_files=8)

    print("\n" + "=" * 120)
    print("INSTANCE:", row["instance_id"])
    print("REPO:", row["repo"])
    print("GOLD FILES:", row["gold_files"])
    print("\nISSUE SNIPPET:")
    print((row["problem_statement"] or "")[:700])

    print("\nSYMBOL CANDIDATES:")
    display(pd.DataFrame(symbol_ranked)[["id", "tail", "score"]])

    print("\nPREDICTED FILES:")
    display(pd.DataFrame(file_ranked))

File-level misses at @5: 18


,instance_id,repo,gold_files,pred_files
0,rich_4077,Textualize/rich,[rich/file_proxy.py],"[rich/progress.py, rich/__init__.py, rich/_nul..."
1,fastapi_14455,fastapi/fastapi,[fastapi/openapi/utils.py],"[fastapi/security/oauth2.py, fastapi/security/..."
2,fastapi_14430,fastapi/fastapi,[fastapi/_compat/v2.py],"[fastapi/routing.py, fastapi/__init__.py, fast..."
3,fastapi_11194,fastapi/fastapi,[fastapi/dependencies/utils.py],"[fastapi/param_functions.py, fastapi/__init__...."
5,fastapi_14482,fastapi/fastapi,[fastapi/_compat/v2.py],"[fastapi/applications.py, fastapi/__init__.py,..."
6,fastapi_13713,fastapi/fastapi,"[fastapi/applications.py, fastapi/openapi/util...","[fastapi/openapi/models.py, fastapi/openapi/__..."
7,rich_3480,Textualize/rich,[rich/text.py],"[rich/protocol.py, rich/__init__.py, rich/cons..."
8,fastapi_12942,fastapi/fastapi,[fastapi/_compat.py],"[fastapi/security/api_key.py, fastapi/security..."
10,fastapi_14356,fastapi/fastapi,[fastapi/dependencies/utils.py],"[fastapi/param_functions.py, fastapi/__init__...."
12,requests_7502,psf/requests,[src/requests/models.py],"[requests/structures.py, requests/__init__.py,..."



INSTANCE: rich_4077
REPO: Textualize/rich
GOLD FILES: ['rich/file_proxy.py']

ISSUE SNIPPET:
proxy isatty

Fixes https://github.com/Textualize/rich/issues/4041

SYMBOL CANDIDATES:


,id,tail,score
0,rich.progress._Reader.isatty,isatty,0.3971
1,rich._null_file.NullFile.isatty,isatty,0.3657
2,rich.progress._Reader,_Reader,0.2172
3,rich._null_file.NullFile,NullFile,0.2081
4,rich.jupyter.print,print,0.1482
5,rich.get_console,get_console,0.1019
6,rich._null_file.NullFile.__enter__,__enter__,0.0690
7,rich._null_file.NullFile.__exit__,__exit__,0.0690
8,rich._null_file.NullFile.__iter__,__iter__,0.0690
9,rich._null_file.NullFile.__next__,__next__,0.0690



PREDICTED FILES:


...

,file,from_symbol,score
0,fastapi/param_functions.py,fastapi.param_functions.File,0.2165
1,fastapi/__init__.py,fastapi.param_functions.File,0.2165
2,fastapi/params.py,fastapi.params.File,0.1799
3,fastapi/openapi/models.py,fastapi.openapi.models.Parameter,0.1187
4,fastapi/openapi/__init__.py,fastapi.openapi.models.Parameter,0.1187
5,fastapi/datastructures.py,fastapi.datastructures.UploadFile,0.0975
6,fastapi/security/oauth2.py,fastapi.security.oauth2.OAuth2PasswordRequestF...,0.0921
7,fastapi/security/__init__.py,fastapi.security.oauth2.OAuth2PasswordRequestF...,0.0921



INSTANCE: fastapi_14482
REPO: fastapi/fastapi
GOLD FILES: ['fastapi/_compat/v2.py']

ISSUE SNIPPET:
🐛 Fix handling arbitrary types when using `arbitrary_types_allowed=True`

🐛 Fix handling arbitrary types when using `arbitrary_types_allowed=True`

Related to https://github.com/fastapi/fastapi/discussions/14184, https://github.com/fastapi/fastapi/issues/14483



SYMBOL CANDIDATES:


,id,tail,score
0,fastapi.applications.FastAPI,FastAPI,0.0765
1,fastapi.exceptions.HTTPException.__init__,__init__,0.0758
2,fastapi.security.http.HTTPBase.make_not_authen...,make_not_authenticated_error,0.0758
3,fastapi.security.oauth2.OAuth2.make_not_authen...,make_not_authenticated_error,0.0758
4,fastapi.applications.FastAPI.exception_handler,exception_handler,0.0756
5,fastapi.exceptions.HTTPException,HTTPException,0.0737
6,fastapi._compat.main._is_model_class,_is_model_class,0.0723
7,fastapi.dependencies.utils._should_embed_body_...,_should_embed_body_fields,0.0723
8,fastapi.params.Header,Header,0.0656
9,fastapi.temp_pydantic_v1_params.Header,Header,0.0656



PREDICTED FILES:


,file,from_symbol,score
0,fastapi/applications.py,fastapi.applications.FastAPI,0.0765
1,fastapi/__init__.py,fastapi.applications.FastAPI,0.0765
2,fastapi/exceptions.py,fastapi.exceptions.HTTPException.__init__,0.0758
3,fastapi/security/http.py,fastapi.security.http.HTTPBase.make_not_authen...,0.0758
4,fastapi/security/__init__.py,fastapi.security.http.HTTPBase.make_not_authen...,0.0758
5,fastapi/security/oauth2.py,fastapi.security.oauth2.OAuth2.make_not_authen...,0.0758
6,fastapi/_compat/main.py,fastapi._compat.main._is_model_class,0.0723
7,fastapi/_compat/__init__.py,fastapi._compat.main._is_model_class,0.0723


Cell 30 — Replay helpers

In [33]:
# Cell 30 — Safer replay helpers
import os
import shutil
import subprocess
import tarfile
from pathlib import Path

REPLAY_ROOT = WORKING / "replay"
REPLAY_ROOT.mkdir(parents=True, exist_ok=True)

def get_task_row(instance_id: str):
    rows = df[df["instance_id"] == instance_id]
    assert len(rows) == 1, f"Expected exactly one row for {instance_id}, got {len(rows)}"
    return rows.iloc[0]

def get_snapshot_path(row):
    return SNAPSHOTS_DIR / f"{row['instance_id']}.tgz"

def extract_snapshot(row, target_dir: Path):
    if target_dir.exists():
        shutil.rmtree(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)

    tpath = get_snapshot_path(row)
    assert tpath.exists(), f"Snapshot not found: {tpath}"

    with tarfile.open(tpath, "r:gz") as tf:
        tf.extractall(target_dir)

    return target_dir

def expected_repo_paths(row):
    """
    Paths that should exist in the repo before applying test_patch.
    Prefer gold source files and source files touched by public patch.
    """
    candidates = []

    # Gold files from earlier patch parser
    if "gold_files" in row and isinstance(row["gold_files"], list):
        candidates.extend(row["gold_files"])

    # Source files from raw patch text
    try:
        for f in parse_patch(row.get("patch", "")):
            if not is_test_path(f["path"]):
                candidates.append(f["path"])
    except Exception:
        pass

    # de-dup preserve order
    out = []
    seen = set()
    for c in candidates:
        if c and c not in seen:
            seen.add(c)
            out.append(c)
    return out

def infer_repo_root_from_task(row, extracted_dir: Path):
    """
    Find the repo root by locating a known repo-relative source path
    somewhere inside the extracted snapshot.
    """
    rel_paths = expected_repo_paths(row)

    for rel in rel_paths:
        rel = str(rel)
        rel_parts = Path(rel).parts
        needle = Path(rel).name

        for p in extracted_dir.rglob(needle):
            p_str = str(p)
            if p_str.endswith(rel):
                root = p
                for _ in rel_parts:
                    root = root.parent
                return root

    # fallback: look for .git
    git_dirs = list(extracted_dir.rglob(".git"))
    if git_dirs:
        return git_dirs[0].parent

    # fallback: look for Python packaging markers
    for marker in ["pyproject.toml", "setup.cfg", "requirements.txt", "setup.py"]:
        hits = list(extracted_dir.rglob(marker))
        if hits:
            return hits[0].parent

    return extracted_dir

def preview_extracted_tree(root: Path, limit=60):
    print(f"\nPreview under: {root}")
    count = 0
    for p in sorted(root.rglob("*")):
        rel = p.relative_to(root)
        print(rel)
        count += 1
        if count >= limit:
            print("... truncated ...")
            break

def ensure_wheels_symlink():
    wheels_link = Path("/wheels")
    if wheels_link.exists():
        print("/wheels already exists:", wheels_link)
        return
    try:
        os.symlink(WHEELS_DIR, wheels_link)
        print("Created /wheels symlink ->", WHEELS_DIR)
    except Exception as e:
        print("Could not create /wheels symlink:", e)

def make_env():
    env = os.environ.copy()
    extra = str(DOCKER_DIR)
    env["PYTHONPATH"] = extra + (":" + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")
    return env

def run_cmd(cmd, cwd, env=None, timeout=1800, check=False):
    print(f"\n$ {' '.join(map(str, cmd))}")
    result = subprocess.run(
        list(map(str, cmd)),
        cwd=str(cwd),
        env=env,
        text=True,
        capture_output=True,
        timeout=timeout,
    )
    print(result.stdout[:5000])
    if result.stderr:
        print("STDERR:")
        print(result.stderr[:3000])
    print("RETURN CODE:", result.returncode)
    if check:
        result.check_returncode()
    return result

def normalize_patch_text(patch_text: str) -> str:
    text = (patch_text or "").replace("\r\n", "\n").replace("\r", "\n")
    if not text.endswith("\n"):
        text += "\n"
    return text

def patch_line_summary(name: str, patch_text: str, head=8, tail=8):
    lines = normalize_patch_text(patch_text).splitlines()
    print(f"\n{name}: {len(lines)} lines")
    print("--- HEAD ---")
    for line in lines[:head]:
        print(line)
    print("--- TAIL ---")
    for line in lines[-tail:]:
        print(line)

def write_patch_file(repo_root: Path, name: str, patch_text: str):
    patch_path = repo_root / f"__{name}.patch"
    patch_path.write_bytes(normalize_patch_text(patch_text).encode("utf-8"))
    return patch_path

def apply_patch_check(repo_root: Path, patch_text: str, name: str):
    patch_path = write_patch_file(repo_root, name, patch_text)
    return run_cmd(
        ["git", "apply", "--check", "--verbose", str(patch_path)],
        cwd=repo_root,
        env=make_env(),
        timeout=300,
        check=False,
    )

def apply_patch(repo_root: Path, patch_text: str, name: str):
    patch_path = write_patch_file(repo_root, name, patch_text)
    return run_cmd(
        ["git", "apply", "--verbose", "--whitespace=nowarn", str(patch_path)],
        cwd=repo_root,
        env=make_env(),
        timeout=300,
        check=False,
    )

def extract_test_targets(test_patch_text: str):
    files = parse_patch(test_patch_text)
    return [f["path"] for f in files if is_test_path(f["path"])]

print("Safer replay helpers ready.")

Safer replay helpers ready.


Cell 31 — Pick one public task and prepare baseline workspace

In [34]:
# Cell 31 — Prepare baseline workspace with stronger repo-root detection
INSTANCE_ID = "fastapi_11194"

task_row = get_task_row(INSTANCE_ID)
print("INSTANCE:", task_row["instance_id"])
print("REPO:", task_row["repo"])
print("EXPECTED SOURCE PATHS:", expected_repo_paths(task_row)[:10])

baseline_dir = REPLAY_ROOT / f"{INSTANCE_ID}_baseline"
extract_snapshot(task_row, baseline_dir)

preview_extracted_tree(baseline_dir, limit=40)

repo_root_baseline = infer_repo_root_from_task(task_row, baseline_dir)

print("\nBaseline workspace:", baseline_dir)
print("Detected repo root:", repo_root_baseline)

# sanity check whether repo-root-relative expected files exist
for rel in expected_repo_paths(task_row)[:10]:
    print(rel, "->", (repo_root_baseline / rel).exists())

INSTANCE: fastapi_11194
REPO: fastapi/fastapi
EXPECTED SOURCE PATHS: ['fastapi/dependencies/utils.py']


/tmp/ipykernel_58/2382390609.py:28: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(target_dir)



Preview under: /kaggle/working/replay/fastapi_11194_baseline
.git
.git/HEAD
.git/config
.git/description
.git/hooks
.git/hooks/applypatch-msg.sample
.git/hooks/commit-msg.sample
.git/hooks/fsmonitor-watchman.sample
.git/hooks/post-update.sample
.git/hooks/pre-applypatch.sample
.git/hooks/pre-commit.sample
.git/hooks/pre-merge-commit.sample
.git/hooks/pre-push.sample
.git/hooks/pre-rebase.sample
.git/hooks/pre-receive.sample
.git/hooks/prepare-commit-msg.sample
.git/hooks/push-to-checkout.sample
.git/hooks/sendemail-validate.sample
.git/hooks/update.sample
.git/index
.git/info
.git/info/exclude
.git/logs
.git/logs/HEAD
.git/logs/refs
.git/logs/refs/heads
.git/logs/refs/heads/export_ref
.git/logs/refs/heads/main
.git/objects
.git/objects/info
.git/objects/pack
.git/objects/pack/pack-ed645e522acb6c14912576866cf0a6e3a0df551e.idx
.git/objects/pack/pack-ed645e522acb6c14912576866cf0a6e3a0df551e.pack
.git/refs
.git/refs/heads
.git/refs/heads/export_ref
.git/refs/heads/main
.git/refs/tags
.git

Cell 32 — Run sandbox setup, apply test_patch, and verify failure

In [35]:
# Cell 32 — Patch-application smoke test before pytest
env = make_env()
ensure_wheels_symlink()

print("\nPatch summaries:")
patch_line_summary("PUBLIC PATCH", task_row["patch"])
patch_line_summary("TEST PATCH", task_row["test_patch"])

# Optional setup; don't hard fail yet
setup_result = run_cmd(
    ["python", str(SANDBOX_DIR / "setup.py")],
    cwd=repo_root_baseline,
    env=env,
    timeout=1800,
    check=False,
)

test_targets = extract_test_targets(task_row["test_patch"])
print("Test targets from test_patch:", test_targets)

check_test_patch = apply_patch_check(repo_root_baseline, task_row["test_patch"], "test_patch")
apply_test_patch_result = apply_patch(repo_root_baseline, task_row["test_patch"], "test_patch")

print("\nAfter applying test patch, do target files exist?")
for t in test_targets:
    print(t, "->", (repo_root_baseline / t).exists())

Created /wheels symlink -> /kaggle/input/competitions/gemma-4-developer-agent/wheels

Patch summaries:

PUBLIC PATCH: 26 lines
--- HEAD ---
--- a/fastapi/dependencies/utils.py
+++ b/fastapi/dependencies/utils.py
@@ -872,20 +872,19 @@ async def _extract_form_body(
     received_body: FormData,
 ) -> Dict[str, Any]:
     values = {}
-    first_field = body_fields[0]
-    first_field_info = first_field.field_info
--- TAIL ---
             value = await value.read()
         elif (
             is_bytes_sequence_field(field)
-            and isinstance(first_field_info, params.File)
+            and isinstance(field_info, params.File)
             and value_is_sequence(value)
         ):
             # For types

TEST PATCH: 93 lines
--- HEAD ---
--- a/tests/test_file_and_form_order_issue_9116.py
+++ b/tests/test_file_and_form_order_issue_9116.py
@@ -0,0 +1,90 @@
+"""
+Regression test, Error 422 if Form is declared before File
+See https://github.com/tiangolo/fastapi/discussions/9116
+"""


Cell 33 — diagnostics

In [36]:
# Cell 33 — Gold patch diagnostics only (no pytest yet)
gold_dir = REPLAY_ROOT / f"{INSTANCE_ID}_gold"
extract_snapshot(task_row, gold_dir)

repo_root_gold = infer_repo_root_from_task(task_row, gold_dir)
print("Detected gold repo root:", repo_root_gold)

for rel in expected_repo_paths(task_row):
    print(rel, "->", (repo_root_gold / rel).exists())

ensure_wheels_symlink()
print("/wheels exists:", Path("/wheels").exists())
print("wheel count:", len(list(Path("/wheels").glob("*.whl"))) if Path("/wheels").exists() else 0)

setup_result_gold = run_cmd(
    ["python", str(SANDBOX_DIR / "setup.py")],
    cwd=repo_root_gold,
    env=env,
    timeout=1800,
    check=False,
)

def numbered_patch_preview(name, patch_text, max_lines=140):
    lines = normalize_patch_text(patch_text).splitlines()
    print(f"\n{name}: total lines = {len(lines)}")
    for i, line in enumerate(lines[:max_lines], 1):
        print(f"{i:03d}: {line}")
    if len(lines) > max_lines:
        print("... truncated ...")

numbered_patch_preview("PUBLIC PATCH", task_row["patch"], max_lines=80)
numbered_patch_preview("TEST PATCH", task_row["test_patch"], max_lines=120)

gold_patch_path = write_patch_file(repo_root_gold, "gold_patch", task_row["patch"])
test_patch_path2 = write_patch_file(repo_root_gold, "test_patch", task_row["test_patch"])

print("\nChecking public gold patch with --recount:")
check_gold_patch = run_cmd(
    ["git", "apply", "--check", "--verbose", "--recount", str(gold_patch_path)],
    cwd=repo_root_gold,
    env=env,
    timeout=300,
    check=False,
)

print("\nChecking test patch with --recount:")
check_test_patch2 = run_cmd(
    ["git", "apply", "--check", "--verbose", "--recount", str(test_patch_path2)],
    cwd=repo_root_gold,
    env=env,
    timeout=300,
    check=False,
)

/tmp/ipykernel_58/2382390609.py:28: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(target_dir)


Detected gold repo root: /kaggle/working/replay/fastapi_11194_gold
fastapi/dependencies/utils.py -> True
/wheels already exists: /wheels
/wheels exists: True
wheel count: 124

$ python /kaggle/input/competitions/gemma-4-developer-agent/sandbox/setup.py
Looking in links: /wheels
Processing /wheels/pytest-9.1.1-py3-none-any.whl
Processing /wheels/typing_extensions-4.16.0-py3-none-any.whl
Processing /wheels/pydantic-2.13.4-py3-none-any.whl
Processing /wheels/sqlmodel-0.0.39-py3-none-any.whl
Processing /wheels/starlette-1.6.0-py3-none-any.whl
Processing /wheels/jinja2-3.1.6-py3-none-any.whl
Processing /wheels/itsdangerous-2.2.0-py3-none-any.whl
Processing /wheels/httpx-0.28.1-py3-none-any.whl
Processing /wheels/anyio-4.14.2-py3-none-any.whl
Processing /wheels/typer-0.26.7-py3-none-any.whl
Processing /wheels/flask-3.1.3-py3-none-any.whl
pytest is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
typing-extensions is a

Cell 34 — apply-only version

In [37]:
# Cell 34 — Apply only if checks pass
apply_gold_patch_result = None
apply_test_patch_result2 = None

if check_gold_patch.returncode == 0:
    print("\nApplying public gold patch...")
    apply_gold_patch_result = run_cmd(
        ["git", "apply", "--verbose", "--recount", "--whitespace=nowarn", str(gold_patch_path)],
        cwd=repo_root_gold,
        env=env,
        timeout=300,
        check=False,
    )
else:
    print("\nSkipping gold patch apply because --check failed.")

if check_test_patch2.returncode == 0:
    print("\nApplying test patch...")
    apply_test_patch_result2 = run_cmd(
        ["git", "apply", "--verbose", "--recount", "--whitespace=nowarn", str(test_patch_path2)],
        cwd=repo_root_gold,
        env=env,
        timeout=300,
        check=False,
    )
else:
    print("\nSkipping test patch apply because --check failed.")

print("\nPost-apply file existence:")
for t in test_targets:
    print(t, "->", (repo_root_gold / t).exists())

print("\nGit status after attempted apply:")
git_status_gold = run_cmd(
    ["git", "status", "--short"],
    cwd=repo_root_gold,
    env=env,
    timeout=120,
    check=False,
)

print("\n=== GOLD PATCH DIAGNOSTIC SUMMARY ===")
print("setup gold rc:", setup_result_gold.returncode)
print("gold patch --check rc:", check_gold_patch.returncode)
print("test patch --check rc:", check_test_patch2.returncode)
print("gold patch apply rc:", None if apply_gold_patch_result is None else apply_gold_patch_result.returncode)
print("test patch apply rc:", None if apply_test_patch_result2 is None else apply_test_patch_result2.returncode)


Applying public gold patch...

$ git apply --verbose --recount --whitespace=nowarn /kaggle/working/replay/fastapi_11194_gold/__gold_patch.patch

STDERR:
Checking patch fastapi/dependencies/utils.py...
Applied patch fastapi/dependencies/utils.py cleanly.

RETURN CODE: 0

Applying test patch...

$ git apply --verbose --recount --whitespace=nowarn /kaggle/working/replay/fastapi_11194_gold/__test_patch.patch

STDERR:
Checking patch tests/test_file_and_form_order_issue_9116.py...
Applied patch tests/test_file_and_form_order_issue_9116.py cleanly.

RETURN CODE: 0

Post-apply file existence:
tests/test_file_and_form_order_issue_9116.py -> True

Git status after attempted apply:

$ git status --short

STDERR:
fatal: detected dubious ownership in repository at '/kaggle/working/replay/fastapi_11194_gold'
To add an exception for this directory, call:

	git config --global --add safe.directory /kaggle/working/replay/fastapi_11194_gold

RETURN CODE: 128

=== GOLD PATCH DIAGNOSTIC SUMMARY ===
setup

Cell 35 — fresh local-deps replay helpers

In [47]:
# Cell 35 — Fresh replay helpers using local deps directory (no venv)
import os
import sys
import shutil
from pathlib import Path

def extract_fresh_repo(instance_id: str, tag: str):
    row = get_task_row(instance_id)
    workdir = REPLAY_ROOT / f"{instance_id}_{tag}"
    extract_snapshot(row, workdir)
    repo_root = infer_repo_root_from_task(row, workdir)

    _ = run_cmd(
        ["git", "config", "--global", "--add", "safe.directory", str(repo_root)],
        cwd=repo_root,
        env=os.environ.copy(),
        timeout=120,
        check=False,
    )
    return row, workdir, repo_root

def make_local_deps_dir(repo_root: Path, label: str):
    deps_dir = repo_root / f".deps_{label}"
    if deps_dir.exists():
        shutil.rmtree(deps_dir, ignore_errors=True)
    deps_dir.mkdir(parents=True, exist_ok=True)
    return deps_dir

def install_local_test_deps(repo_root: Path, deps_dir: Path):
    ensure_wheels_symlink()

    # Use wheel-resolvable package specs from the competition bundle
    specs = [
        "pytest",
        "httpx",
        "anyio",
        "typing_extensions",
        "pydantic<3",
        "starlette<0.37",
    ]

    # Optional multipart support if the wheel exists
    multipart_candidates = list(WHEELS_DIR.glob("python_multipart*.whl")) + list(WHEELS_DIR.glob("python-multipart*.whl"))
    if multipart_candidates:
        specs.append("python-multipart")

    print("Installing local deps:", specs)

    return run_cmd(
        [
            sys.executable, "-m", "pip", "install",
            "--target", str(deps_dir),
            "--no-index",
            "--find-links", "/wheels",
            *specs
        ],
        cwd=repo_root,
        env=os.environ.copy(),
        timeout=1800,
        check=False,
    )

def make_repo_test_env(repo_root: Path, deps_dir: Path):
    env = os.environ.copy()
    env["PYTHONNOUSERSITE"] = "1"
    env["PYTHONDONTWRITEBYTECODE"] = "1"
    env["PYTHONPATH"] = os.pathsep.join([
        str(repo_root),   # import repo code first
        str(deps_dir),    # import local wheel-installed deps second
        str(DOCKER_DIR),  # imp.py / telnetlib.py compatibility shims
    ])
    return env

def import_sanity_local(repo_root: Path, deps_dir: Path, module_name: str):
    code = f"""
import sys
import {module_name}
print("{module_name} imported from:", {module_name}.__file__)
try:
    import starlette
    print("starlette imported from:", starlette.__file__)
except Exception as e:
    print("starlette import failed:", e)
print("sys.path[:8] =", sys.path[:8])
"""
    return run_cmd(
        [sys.executable, "-S", "-c", code],
        cwd=repo_root,
        env=make_repo_test_env(repo_root, deps_dir),
        timeout=120,
        check=False,
    )

print("Fresh replay helpers loaded.")

Fresh replay helpers loaded.


Cell 36 — baseline isolated by local deps

In [48]:
# Cell 36 — Baseline replay with local deps
INSTANCE_ID = "fastapi_11194"

task_row, baseline_dir, repo_root_baseline = extract_fresh_repo(INSTANCE_ID, "baseline_localdeps")
baseline_deps_dir = make_local_deps_dir(repo_root_baseline, "baseline")

print("Baseline repo root:", repo_root_baseline)
print("Baseline deps dir:", baseline_deps_dir)

baseline_dep_install = install_local_test_deps(repo_root_baseline, baseline_deps_dir)

test_targets = extract_test_targets(task_row["test_patch"])
print("Test targets:", test_targets)

baseline_test_check = apply_patch_check(repo_root_baseline, task_row["test_patch"], "baseline_test_patch")
baseline_test_apply = apply_patch(repo_root_baseline, task_row["test_patch"], "baseline_test_patch")

print("\nRunning BASELINE import sanity")
baseline_import_check = import_sanity_local(repo_root_baseline, baseline_deps_dir, "fastapi")

print("\nRunning BASELINE pytest")
baseline_pytest_clean = run_cmd(
    [sys.executable, "-S", "-m", "pytest", "-q", *test_targets, "--maxfail=1"],
    cwd=repo_root_baseline,
    env=make_repo_test_env(repo_root_baseline, baseline_deps_dir),
    timeout=1800,
    check=False,
)

print("\nExpected: baseline should FAIL")
print("Baseline pytest rc:", baseline_pytest_clean.returncode)

/tmp/ipykernel_58/2382390609.py:28: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(target_dir)



$ git config --global --add safe.directory /kaggle/working/replay/fastapi_11194_baseline_localdeps

RETURN CODE: 0
Baseline repo root: /kaggle/working/replay/fastapi_11194_baseline_localdeps
Baseline deps dir: /kaggle/working/replay/fastapi_11194_baseline_localdeps/.deps_baseline
/wheels already exists: /wheels
Installing local deps: ['pytest', 'httpx', 'anyio', 'typing_extensions', 'pydantic<3', 'starlette<0.37']

$ /usr/bin/python3 -m pip install --target /kaggle/working/replay/fastapi_11194_baseline_localdeps/.deps_baseline --no-index --find-links /wheels pytest httpx anyio typing_extensions pydantic<3 starlette<0.37
Looking in links: /wheels
Processing /wheels/pytest-9.1.1-py3-none-any.whl
Processing /wheels/httpx-0.28.1-py3-none-any.whl
Processing /wheels/anyio-4.14.2-py3-none-any.whl
Processing /wheels/typing_extensions-4.16.0-py3-none-any.whl
Processing /wheels/pydantic-2.13.4-py3-none-any.whl
Processing /wheels/starlette-0.36.0-py3-none-any.whl
Processing /wheels/iniconfig-2.3

Cell 37 — gold isolated by local deps

In [49]:
# Cell 37 — Gold replay with local deps
task_row, gold_dir, repo_root_gold = extract_fresh_repo(INSTANCE_ID, "gold_localdeps")
gold_deps_dir = make_local_deps_dir(repo_root_gold, "gold")

print("Gold repo root:", repo_root_gold)
print("Gold deps dir:", gold_deps_dir)

gold_dep_install = install_local_test_deps(repo_root_gold, gold_deps_dir)

gold_patch_check = apply_patch_check(repo_root_gold, task_row["patch"], "gold_patch")
gold_patch_apply = apply_patch(repo_root_gold, task_row["patch"], "gold_patch")

gold_test_check = apply_patch_check(repo_root_gold, task_row["test_patch"], "gold_test_patch")
gold_test_apply = apply_patch(repo_root_gold, task_row["test_patch"], "gold_test_patch")

print("\nRunning GOLD import sanity")
gold_import_check = import_sanity_local(repo_root_gold, gold_deps_dir, "fastapi")

print("\nRunning GOLD pytest")
gold_pytest_clean = run_cmd(
    [sys.executable, "-S", "-m", "pytest", "-q", *test_targets, "--maxfail=1"],
    cwd=repo_root_gold,
    env=make_repo_test_env(repo_root_gold, gold_deps_dir),
    timeout=1800,
    check=False,
)

print("\nExpected: gold should PASS")
print("Gold pytest rc:", gold_pytest_clean.returncode)

/tmp/ipykernel_58/2382390609.py:28: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(target_dir)



$ git config --global --add safe.directory /kaggle/working/replay/fastapi_11194_gold_localdeps

RETURN CODE: 0
Gold repo root: /kaggle/working/replay/fastapi_11194_gold_localdeps
Gold deps dir: /kaggle/working/replay/fastapi_11194_gold_localdeps/.deps_gold
/wheels already exists: /wheels
Installing local deps: ['pytest', 'httpx', 'anyio', 'typing_extensions', 'pydantic<3', 'starlette<0.37']

$ /usr/bin/python3 -m pip install --target /kaggle/working/replay/fastapi_11194_gold_localdeps/.deps_gold --no-index --find-links /wheels pytest httpx anyio typing_extensions pydantic<3 starlette<0.37
Looking in links: /wheels
Processing /wheels/pytest-9.1.1-py3-none-any.whl
Processing /wheels/httpx-0.28.1-py3-none-any.whl
Processing /wheels/anyio-4.14.2-py3-none-any.whl
Processing /wheels/typing_extensions-4.16.0-py3-none-any.whl
Processing /wheels/pydantic-2.13.4-py3-none-any.whl
Processing /wheels/starlette-0.36.0-py3-none-any.whl
Processing /wheels/iniconfig-2.3.0-py3-none-any.whl (from pytest

Cell 38 — replay verdict

In [50]:
# Cell 38 — Local-deps replay verdict
baseline_rc = baseline_pytest_clean.returncode if baseline_pytest_clean is not None else None
gold_rc = gold_pytest_clean.returncode if gold_pytest_clean is not None else None

print("=== LOCAL-DEPS REPLAY VERDICT ===")
print("Instance:", INSTANCE_ID)

print("baseline deps install rc:", baseline_dep_install.returncode)
print("gold deps install rc:", gold_dep_install.returncode)

print("baseline test --check rc:", baseline_test_check.returncode)
print("baseline test apply rc:", baseline_test_apply.returncode)

print("gold patch --check rc:", gold_patch_check.returncode)
print("gold patch apply rc:", gold_patch_apply.returncode)
print("gold test --check rc:", gold_test_check.returncode)
print("gold test apply rc:", gold_test_apply.returncode)

print("baseline import rc:", baseline_import_check.returncode)
print("gold import rc:", gold_import_check.returncode)

print("baseline pytest rc:", baseline_rc)
print("gold pytest rc:", gold_rc)

if baseline_rc is not None and gold_rc is not None and baseline_rc != 0 and gold_rc == 0:
    print("PASS: local replay matches fail-to-pass / pass-to-pass expectation.")
else:
    print("NOT CLEAN YET: inspect dependency install and pytest outputs above.")

=== LOCAL-DEPS REPLAY VERDICT ===
Instance: fastapi_11194
baseline deps install rc: 1
gold deps install rc: 1
baseline test --check rc: 0
baseline test apply rc: 0
gold patch --check rc: 0
gold patch apply rc: 0
gold test --check rc: 0
gold test apply rc: 0
baseline import rc: 1
gold import rc: 1
baseline pytest rc: 1
gold pytest rc: 1
NOT CLEAN YET: inspect dependency install and pytest outputs above.


Cell 39 — if replay works, build decision

In [51]:
# Cell 39 — Decide next step
if baseline_pytest_clean.returncode != 0 and gold_pytest_clean.returncode == 0:
    print("Replay works. Next step: move to submission-building track.")
else:
    print("Replay still not clean. Fix replay before submission.")

Replay still not clean. Fix replay before submission.
